# 1 - Caracterização da Amostra

In [ ]:
import json
import csv
from collections import defaultdict

# Caminho dos arquivos
MODULOS_PATH = "output/mapeamento_provas_questoes.json"
QUESTIONS_PATH = "output/dataset_analise_questoes.csv"
MISCONCEPTIONS_PATH = "output/misconceptions_detalhado_por_usuario.csv"

#Codigos Auxiliares

def carregar_estrutura_completa():
    """Carrega o arquivo JSON com a estrutura completa das provas"""
    with open(MODULOS_PATH, 'r', encoding='utf-8') as f:
        return json.load(f)

def contar_questoes_totais(estrutura):
    """Conta o número total de questões únicas"""
    questoes = set()
    
    for disciplina, provas in estrutura.get('mapeamento_provas', {}).items():
        for prova_nome, prova_data in provas.items():
            for turma_id, turma_info in prova_data.items():
                if 'info' in turma_info and 'exercises' in turma_info['info']:
                    for exercicio, questoes_lista in turma_info['info']['exercises'].items():
                        questoes.update(questoes_lista)
    
    return len(questoes)

def contar_turmas(estrutura):
    """Conta o número de turmas considerando período e exclusões"""
    turmas = set()
    exclusoes = ['2020-ERE', '2020/1', '2020/2']
    
    for disciplina, provas in estrutura.get('mapeamento_provas', {}).items():
        for prova_nome, prova_data in provas.items():
            for turma_id, turma_info in prova_data.items():
                if 'info' in turma_info:
                    info = turma_info['info']
                    start_date = info.get('start', '')
                    
                    # Extrai ano e semestre da data
                    if start_date:
                        ano = int(start_date.split('-')[0])
                        mes = int(start_date.split('-')[1])
                        semestre = 1 if mes <= 6 else 2
                        periodo = f"{ano}/{semestre}"
                        
                        # Verifica se está no período válido (2016/1 a 2024/1)
                        if ano < 2016 or (ano == 2016 and semestre < 1):
                            continue
                        if ano > 2024 or (ano == 2024 and semestre > 1):
                            continue
                        
                        # Verifica exclusões
                        if periodo in exclusoes or '2020-ERE' in start_date:
                            continue
                        
                        turmas.add((info.get('class_number'), periodo))
    
    return len(turmas)

def contar_questoes_16_mais_respostas():
    """
    Conta questões com 16 ou mais respostas
    
    COMO FUNCIONA:
    - Lê o arquivo output/dataset_analise_questoes.csv que contém questões com 16+ respostas
    - Conta o número de linhas (cada linha = uma questão)
    - Retorna o total de questões
    """
    questoes_validas = 0
    
    try:
        with open(QUESTIONS_PATH, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            
            for linha in reader:
                questoes_validas += 1
        
        return questoes_validas
    except FileNotFoundError:
        print(f"Arquivo {QUESTIONS_PATH} não encontrado")
        return 0

def contar_estudantes():
    """Conta o número de estudantes únicos"""
    estudantes = set()
    
    try:
        with open(MISCONCEPTIONS_PATH, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            
            for linha in reader:
                if 'usuario' in linha and linha['usuario']:
                    estudantes.add(linha['usuario'])
        
        return len(estudantes)
    except FileNotFoundError:
        print(f"Arquivo {MISCONCEPTIONS_PATH} não encontrado")
        return 0

def contar_codigos_analisados():
    """
    Conta o número total de códigos analisados
    
    COMO FUNCIONA:
    - Lê o arquivo output/dataset_analise_questoes.csv
    - Para cada questão, pega a coluna 'respostas'
    - Essa coluna já contém o número de respostas para aquela questão
    - Soma todos os valores para obter o total de códigos individuais analisados
    
    EXEMPLO:
    respostas: 42
    -> 42 códigos analisados para essa questão
    """
    total_codigos = 0
    
    try:
        with open(QUESTIONS_PATH, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            
            for linha in reader:
                if 'respostas' in linha and linha['respostas']:
                    try:
                        num_respostas = int(linha['respostas'])
                        total_codigos += num_respostas
                    except ValueError:
                        # Ignora se não for um número válido
                        continue
        
        return total_codigos
    except FileNotFoundError:
        print(f"Arquivo {QUESTIONS_PATH} não encontrado")
        return 0

def main():
    print("=" * 70)
    print("LEVANTAMENTO DE INFORMAÇÕES")
    print("=" * 70)
    
    # Período
    print("\n PERÍODO DO LEVANTAMENTO:")
    print("   • Ano/Semestre Inicial: 2016/1")
    print("   • Ano/Semestre Final: 2024/1")
    print("   • Exclusões: 2020-ERE, 2020/1, 2020/2")
    
    # Carrega estrutura
    try:
        estrutura = carregar_estrutura_completa()
        
        # Número de turmas
        num_turmas = contar_turmas(estrutura)
        print(f"\n NÚMERO DE TURMAS: {num_turmas}")
        
        # Número de questões totais
        num_questoes_total = contar_questoes_totais(estrutura)
        print(f"\n NÚMERO DE QUESTÕES DE PROGRAMAÇÃO:")
        print(f"   • Total: {num_questoes_total}")
        
    except FileNotFoundError:
        print(f"\n Arquivo {MODULOS_PATH} não encontrado")
        num_turmas = 0
        num_questoes_total = 0
    
    # Questões com 16+ respostas
    num_questoes_16_mais = contar_questoes_16_mais_respostas()
    print(f"   • Respondidas por 16 ou mais estudantes: {num_questoes_16_mais}")
    
    # Número de estudantes
    num_estudantes = contar_estudantes()
    print(f"\n NÚMERO DE ESTUDANTES: {num_estudantes}")
    
    # Número de códigos analisados
    num_codigos = contar_codigos_analisados()
    print(f"\n NÚMERO DE CÓDIGOS ANALISADOS: {num_codigos}")
    
    print("\n" + "=" * 70)

if __name__ == "__main__":
    main()
from pathlib import Path


def carregar_referencias(path):
    references_path = Path(path).resolve()
    data = json.loads(references_path.read_text(encoding='utf-8'))
    source_root = Path(data['source_root'])
    if not source_root.is_absolute():
        source_root = (references_path.parent / source_root).resolve()
    data['_source_root'] = str(source_root)
    data['_references_path'] = str(references_path)
    return data


def iterar_arquivos_usuario(data, user_id, kind, suffix=None):
    source_root = Path(data['_source_root'])
    for record in data.get('users', []):
        if str(record.get('id')) != str(user_id):
            continue
        for item in record.get('files', {}).get(kind, []):
            path = source_root / item['path']
            if path.is_file() and (suffix is None or path.name.endswith(suffix)):
                yield path



# 2 - Análise dos PC³

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
import numpy as np
import locale

# Configurar locale para formatação brasileira
try:
    locale.setlocale(locale.LC_ALL, 'pt_BR.UTF-8')
except:
    pass

# --- CONFIGURAÇÕES GERAIS ---
QUESTIONS_PATH = "output/dataset_analise_questoes.csv"

# Configurações de fonte global
plt.rcParams['font.size'] = 14
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

# Mapeamento de Colunas para Nomes Legíveis
PC3_MAP = {
    'A4': 'A4', 'A5': 'A5', 'B4': 'B4', 'B6': 'B6',
    'B8': 'B8', 'B9': 'B9', 'B12': 'B12', 'C1': 'C1',
    'C2': 'C2', 'C3': 'C3', 'C4': 'C4', 'C8': 'C8',
    'D4': 'D4', 'E2': 'E2', 'G4': 'G4', 'G5': 'G5',
    'H1': 'H1'
}

# Lista base (será filtrada dinamicamente)
PC3_COLUNAS_BASE = list(PC3_MAP.keys())

def formatar_numero(valor):
    """Formata número com separador de milhares (.) e decimais com vírgula"""
    if valor >= 1000:
        return f"{int(valor):,}".replace(',', '.')
    return str(int(valor))

def formatar_decimal(valor):
    """Formata decimal com vírgula"""
    return f"{valor:.2f}".replace('.', ',')

def carregar_dados():
    """Carrega os dados do CSV e trata colunas inexistentes"""
    try:
        df = pd.read_csv(QUESTIONS_PATH)
        # Garante que todas as colunas de interesse existam no DF (preenche com 0 se faltar)
        for col in PC3_COLUNAS_BASE:
            if col not in df.columns:
                df[col] = 0
        return df
    except FileNotFoundError:
        print(f"❌ Erro: Arquivo '{QUESTIONS_PATH}' não encontrado.")
        return None

def obter_colunas_ordenadas(df, top_n=None):
    """
    1. Calcula a frequência total.
    2. Remove colunas zeradas (ex: A4, H1).
    3. Ordena do mais frequente para o menos frequente.
    4. Retorna a lista de colunas cortada pelo top_n.
    """
    totais = {}
    
    # Calcula totais
    for col in PC3_COLUNAS_BASE:
        soma = df[col].sum()
        totais[col] = soma

    # Filtra quem tem 0 ocorrências (Remove A4 e H1 automaticamente se estiverem vazios)
    totais_ativos = {k: v for k, v in totais.items() if v > 0}
    
    # Identifica quem foi removido
    removidos = set(PC3_COLUNAS_BASE) - set(totais_ativos.keys())
    if removidos:
        nomes_removidos = [PC3_MAP[c] for c in removidos]
        print(f"⚠️ PC³s vazios removidos da análise de correlação: {', '.join(nomes_removidos)}")

    # Ordena dicionário por valor (decrescente)
    ordenados = sorted(totais_ativos.items(), key=lambda x: x[1], reverse=True)
    
    # Pega apenas as chaves (nomes das colunas)
    colunas_ordenadas = [item[0] for item in ordenados]
    
    # Aplica o corte top_n se solicitado
    if top_n and top_n < len(colunas_ordenadas):
        print(f"ℹ️ Selecionando os Top {top_n} PC³s mais frequentes.")
        colunas_ordenadas = colunas_ordenadas[:top_n]
    else:
        print(f"ℹ️ Analisando todos os {len(colunas_ordenadas)} PC³s com ocorrências.")
        
    return colunas_ordenadas, totais

def plotar_frequencia(totais, colunas_ordenadas):
    """Plota gráfico de barras seguindo a ordem definida"""
    
    nomes = [PC3_MAP[col] for col in colunas_ordenadas]
    valores = [totais[col] for col in colunas_ordenadas]
    
    fig, ax = plt.subplots(figsize=(14, 7))
    bars = ax.bar(nomes, valores, color='steelblue')
    
    # Adiciona valores no topo com formatação brasileira
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                formatar_numero(height),
                ha='center', va='bottom', fontsize=13, fontweight='bold')
    
    ax.set_xlabel('PC³s', fontsize=16)
    ax.set_ylabel('Frequência absoluta', fontsize=16)
    #ax.set_title('PC³s mais frequentes', fontsize=18, fontweight='bold')
    plt.xticks(rotation=0, ha='center')  # Nomes retos e centralizados
    
    # Formata eixo Y com separador de milhares
    from matplotlib.ticker import FuncFormatter
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_numero(x)))
    
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

def plotar_misconception(df, colunas_ordenadas):
    """Plota gráfico de misconception por questão (em azul)"""
    
    # Verifica se a coluna 'misconception' existe
    if 'misconception' not in df.columns:
        print("⚠️ Coluna 'misconception' não encontrada no dataset.")
        return
    
    # Agrupa por questão e soma misconceptions
    if 'questao' in df.columns:
        misc_por_questao = df.groupby('questao')['misconception'].sum().sort_values(ascending=False)
    elif 'id' in df.columns:
        misc_por_questao = df.groupby('id')['misconception'].sum().sort_values(ascending=False)
    else:
        print("⚠️ Coluna 'questao' ou 'id' não encontrada.")
        return
    
    # Limita aos top 15 para melhor visualização
    top_questoes = misc_por_questao.head(15)
    
    fig, ax = plt.subplots(figsize=(14, 7))
    bars = ax.bar(range(len(top_questoes)), top_questoes.values, color='steelblue')
    
    # Adiciona valores no topo
    for i, (bar, val) in enumerate(zip(bars, top_questoes.values)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                formatar_numero(val),
                ha='center', va='bottom', fontsize=13, fontweight='bold')
    
    ax.set_xlabel('questão', fontsize=16)
    ax.set_ylabel('frequência de misconception', fontsize=16)
    ax.set_title('misconception por questão', fontsize=18, fontweight='bold')
    ax.set_xticks(range(len(top_questoes)))
    plt.xticks(range(len(top_questoes)), top_questoes.index, rotation=0, ha='center')
    
    # Formata eixo Y com separador de milhares
    from matplotlib.ticker import FuncFormatter
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_numero(x)))
    
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

def calcular_e_plotar_correlacao(df, colunas_ordenadas):
    """
    Calcula correlação apenas para as colunas selecionadas e ordenadas.
    """
    # Filtra apenas questões relevantes (respostas >= 16, se coluna existir)
    if 'respostas' in df.columns:
        df_corr = df[df['respostas'] >= 16].copy()
    else:
        df_corr = df.copy()

    # Seleciona subconjunto de dados JÁ ORDENADO por frequência
    df_subset = df_corr[colunas_ordenadas]
    
    # Renomeia para nomes curtos (C1, B4...) para o gráfico ficar bonito
    df_subset.columns = [PC3_MAP[col] for col in colunas_ordenadas]
    
    # Calcula Spearman
    corr_df = df_subset.corr(method='spearman')
    
    # Plotagem com fonte maior
    fig, ax = plt.subplots(figsize=(14, 12))
    
    # Máscara para triângulo superior
    mask = np.triu(np.ones_like(corr_df, dtype=bool))
    
    # Cria anotações formatadas com vírgula
    annot_data = corr_df.applymap(lambda x: formatar_decimal(x))
    
    # MUDANÇA: inverte o colormap - agora azul é positivo e vermelho é negativo
    heatmap = sns.heatmap(
        corr_df,
        mask=mask,
        annot=annot_data,
        fmt='',
        cmap='coolwarm_r',  # INVERTIDO: adiciona _r para reverter as cores
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        linewidths=0.5,
        cbar_kws={'label': 'Correlação de Spearman'},
        ax=ax,
        annot_kws={'fontsize': 12}
    )
    
    # Formata a barra de cores com vírgula nos decimais
    from matplotlib.ticker import FuncFormatter
    colorbar = heatmap.collections[0].colorbar
    colorbar.ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_decimal(x)))
    
    #ax.set_title('matriz de correlação de PC³s (ordenada por frequência)',              fontsize=18, fontweight='bold', pad=20)
    
    # Ajusta labels para ficarem retos e centralizados
    plt.xticks(rotation=0, ha='center')
    plt.yticks(rotation=0, va='center')
    
    plt.tight_layout()
    plt.show()
    
    return corr_df

def gerar_relatorio(top_n=None):
    """
    Função Principal de Controle.
    
    Parâmetros:
      top_n (int ou None): Número de PC³s a analisar. 
                           Ex: 10 para os top 10, None para todos.
    """
    print("=" * 60)
    print("INICIANDO ANÁLISE DE PC³s")
    print("=" * 60)
    
    df = carregar_dados()
    if df is None: return

    # 1. Processamento e Ordenação
    colunas_finais, totais_brutos = obter_colunas_ordenadas(df, top_n=top_n)
    
    if not colunas_finais:
        print("Nenhum dado encontrado para gerar relatórios.")
        return

    # 2. Plotar Frequências de PC³s (Barras)
    print("\n📊 Gerando gráfico de frequências de PC³s...")
    plotar_frequencia(totais_brutos, colunas_finais)
    
    # 3. Plotar Misconception por Questão (Barras Azuis)
    print("\n📊 Gerando gráfico de misconception por questão...")
    plotar_misconception(df, colunas_finais)
    
    # 4. Correlação (Heatmap)
    print("\n🔥 Gerando mapa de calor de correlações entre PC³s...")
    corr_df = calcular_e_plotar_correlacao(df, colunas_finais)
    
    # 5. Exibir informação
    print("\n✓ Matriz de correlação gerada com sucesso!")
    print("=" * 60)
    
    return corr_df


# VERSÃO NOTEBOOK - Chama a função diretamente
gerar_relatorio(top_n=4)  # top_n para a quantidade de PC³s a analisar

# 2 - Versão PDF

In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
import numpy as np
import locale

# ── Texto selecionável no PDF ──────────────────────────────
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42
# ──────────────────────────────────────────────────────────

# Configurar locale para formatação brasileira
try:
    locale.setlocale(locale.LC_ALL, 'pt_BR.UTF-8')
except:
    pass

# --- CONFIGURAÇÕES GERAIS ---
QUESTIONS_PATH = "output/dataset_analise_questoes.csv"

# Configurações de fonte global
plt.rcParams['font.size'] = 14
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

# Mapeamento de Colunas para Nomes Legíveis
PC3_MAP = {
    'A4': 'A4', 'A5': 'A5', 'B4': 'B4', 'B6': 'B6',
    'B8': 'B8', 'B9': 'B9', 'B12': 'B12', 'C1': 'C1',
    'C2': 'C2', 'C3': 'C3', 'C4': 'C4', 'C8': 'C8',
    'D4': 'D4', 'E2': 'E2', 'G4': 'G4', 'G5': 'G5',
    'H1': 'H1'
}

PC3_COLUNAS_BASE = list(PC3_MAP.keys())

def formatar_numero(valor):
    if valor >= 1000:
        return f"{int(valor):,}".replace(',', '.')
    return str(int(valor))

def formatar_decimal(valor):
    return f"{valor:.2f}".replace('.', ',')

def carregar_dados():
    try:
        df = pd.read_csv(QUESTIONS_PATH)
        for col in PC3_COLUNAS_BASE:
            if col not in df.columns:
                df[col] = 0
        return df
    except FileNotFoundError:
        print(f"❌ Erro: Arquivo '{QUESTIONS_PATH}' não encontrado.")
        return None

def obter_colunas_ordenadas(df, top_n=None):
    totais = {}
    for col in PC3_COLUNAS_BASE:
        soma = df[col].sum()
        totais[col] = soma

    totais_ativos = {k: v for k, v in totais.items() if v > 0}
    removidos = set(PC3_COLUNAS_BASE) - set(totais_ativos.keys())
    if removidos:
        nomes_removidos = [PC3_MAP[c] for c in removidos]
        print(f"⚠️ PC³s vazios removidos da análise de correlação: {', '.join(nomes_removidos)}")

    ordenados = sorted(totais_ativos.items(), key=lambda x: x[1], reverse=True)
    colunas_ordenadas = [item[0] for item in ordenados]

    if top_n and top_n < len(colunas_ordenadas):
        print(f"ℹ️ Selecionando os Top {top_n} PC³s mais frequentes.")
        colunas_ordenadas = colunas_ordenadas[:top_n]
    else:
        print(f"ℹ️ Analisando todos os {len(colunas_ordenadas)} PC³s com ocorrências.")

    return colunas_ordenadas, totais

def plotar_frequencia(totais, colunas_ordenadas):
    nomes = [PC3_MAP[col] for col in colunas_ordenadas]
    valores = [totais[col] for col in colunas_ordenadas]

    fig, ax = plt.subplots(figsize=(14, 7))
    bars = ax.bar(nomes, valores, color='steelblue')

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                formatar_numero(height),
                ha='center', va='bottom', fontsize=13, fontweight='bold')

    ax.set_xlabel('PC³s', fontsize=16)
    ax.set_ylabel('Frequência absoluta', fontsize=16)
    plt.xticks(rotation=0, ha='center')

    from matplotlib.ticker import FuncFormatter
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_numero(x)))

    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig("frequencia_pc3.pdf", bbox_inches='tight', format='pdf')
    plt.close()
    print("💾 Salvo: frequencia_pc3.pdf")

def plotar_misconception(df, colunas_ordenadas):
    if 'misconception' not in df.columns:
        print("⚠️ Coluna 'misconception' não encontrada no dataset.")
        return

    if 'questao' in df.columns:
        misc_por_questao = df.groupby('questao')['misconception'].sum().sort_values(ascending=False)
    elif 'id' in df.columns:
        misc_por_questao = df.groupby('id')['misconception'].sum().sort_values(ascending=False)
    else:
        print("⚠️ Coluna 'questao' ou 'id' não encontrada.")
        return

    top_questoes = misc_por_questao.head(15)

    fig, ax = plt.subplots(figsize=(14, 7))
    bars = ax.bar(range(len(top_questoes)), top_questoes.values, color='steelblue')

    for i, (bar, val) in enumerate(zip(bars, top_questoes.values)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                formatar_numero(val),
                ha='center', va='bottom', fontsize=13, fontweight='bold')

    ax.set_xlabel('questão', fontsize=16)
    ax.set_ylabel('frequência de misconception', fontsize=16)
    ax.set_title('misconception por questão', fontsize=18, fontweight='bold')
    ax.set_xticks(range(len(top_questoes)))
    plt.xticks(range(len(top_questoes)), top_questoes.index, rotation=0, ha='center')

    from matplotlib.ticker import FuncFormatter
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_numero(x)))

    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig("misconception_por_questao.pdf", bbox_inches='tight', format='pdf')
    plt.close()
    print("💾 Salvo: misconception_por_questao.pdf")

def calcular_e_plotar_correlacao(df, colunas_ordenadas):
    if 'respostas' in df.columns:
        df_corr = df[df['respostas'] >= 16].copy()
    else:
        df_corr = df.copy()

    df_subset = df_corr[colunas_ordenadas]
    df_subset.columns = [PC3_MAP[col] for col in colunas_ordenadas]

    corr_df = df_subset.corr(method='spearman')

    fig, ax = plt.subplots(figsize=(10, 9))   # figura menor → quadrados menores
    mask = np.triu(np.ones_like(corr_df, dtype=bool))
    annot_data = corr_df.applymap(lambda x: formatar_decimal(x))

    heatmap = sns.heatmap(
        corr_df,
        mask=mask,
        annot=annot_data,
        fmt='',
        cmap='coolwarm_r',
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        linewidths=0.5,
        cbar_kws={'label': 'Correlação de Spearman'},
        ax=ax,
        annot_kws={'fontsize': 16}   # valores dentro das células
    )

    from matplotlib.ticker import FuncFormatter
    colorbar = heatmap.collections[0].colorbar
    colorbar.ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_decimal(x)))
    colorbar.ax.tick_params(labelsize=14)
    colorbar.set_label('Correlação de Spearman', fontsize=16)

    ax.tick_params(axis='both', labelsize=16)      # labels dos eixos X e Y
    plt.xticks(rotation=0, ha='center')
    plt.yticks(rotation=0, va='center')

    plt.tight_layout()
    plt.savefig("correlacao_pc3.pdf", bbox_inches='tight', format='pdf')
    plt.close()
    print("💾 Salvo: correlacao_pc3.pdf")

    return corr_df

def gerar_relatorio(top_n=None):
    print("=" * 60)
    print("INICIANDO ANÁLISE DE PC³s")
    print("=" * 60)

    df = carregar_dados()
    if df is None: return

    colunas_finais, totais_brutos = obter_colunas_ordenadas(df, top_n=top_n)

    if not colunas_finais:
        print("Nenhum dado encontrado para gerar relatórios.")
        return

    print("\n📊 Gerando frequencia_pc3.pdf ...")
    plotar_frequencia(totais_brutos, colunas_finais)

    print("\n📊 Gerando misconception_por_questao.pdf ...")
    plotar_misconception(df, colunas_finais)

    print("\n🔥 Gerando correlacao_pc3.pdf ...")
    corr_df = calcular_e_plotar_correlacao(df, colunas_finais)

    print("\n✓ Matriz de correlação gerada com sucesso!")
    print("=" * 60)

    return corr_df


gerar_relatorio(top_n=4)

# 3 - Análise das métricas de dificuldade

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# Configurações
QUESTIONS_PATH = "output/dataset_analise_questoes.csv"

# Configurações de fonte global
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Mapeamento de Colunas para Nomes Legíveis
PC3_MAP = {
    'A4': 'A4', 'A5': 'A5', 'B4': 'B4', 'B6': 'B6',
    'B8': 'B8', 'B9': 'B9', 'B12': 'B12', 'C1': 'C1',
    'C2': 'C2', 'C3': 'C3', 'C4': 'C4', 'C8': 'C8',
    'D4': 'D4', 'E2': 'E2', 'G4': 'G4', 'G5': 'G5',
    'H1': 'H1'
}

# ============================================================================
# MAPEAMENTO DAS MÉTRICAS DE DIFICULDADE
# Colunas reais do CSV → nomes legíveis
# 'dificuldade' é ignorada conforme instrução
# ============================================================================
# M1:  taxa_acerto           = num_correct / num_students_interactions
# M2:  num_submissoes        = num_submissions / num_students_interactions
# M3:  taxa_aceitacao        = num_correct / num_submissions
# M4:  num_testes            = num_tests / num_students_interactions
# M5:  num_consultas         = (num_submissions + num_tests) / num_students_interactions
# M6:  num_erros_lgcs        = num_logic_errors / num_students_interactions
# M7:  num_errors_stx        = num_syntax_errors / num_students_interactions
# M8:  num_erros             = num_errors / num_students_interactions
# M9:  num_eventos           = num_events / num_students_interactions
# M10: num_eventos_del       = num_deletes / num_students_interactions
# M11: tempo_implementacao   = code_time / num_correct
# M12: qtd_alteracoes_codigo = amount_of_change / num_students_interactions
# M13: discriminacao         (fórmula própria - já calculada)

traducao_metricas = {
    'taxa_acerto':           'taxa de acerto',       # M1
    'num_submissoes':        'submissões',            # M2
    'taxa_aceitacao':        'taxa de aceitação',     # M3
    'num_testes':            'testes',                # M4
    'num_consultas':         'consultas',             # M5
    'num_erros_lgcs':        'erros lógicos',         # M6
    'num_errors_stx':        'erros sintaxe',         # M7
    'num_erros':             'erros totais',          # M8
    'num_eventos':           'eventos totais',        # M9
    'num_eventos_del':       'eventos deleção',       # M10
    'tempo_implementacao':   'tempo implementação',   # M11
    'qtd_alteracoes_codigo': 'alterações código',     # M12
    'discriminacao':         'discriminação',         # M13
    # 'dificuldade' IGNORADA intencionalmente
}

# Métricas onde valores menores = questão mais difícil (positivas invertidas)
# Usadas apenas para referência semântica
METRICAS_POSITIVAS = {'taxa_acerto', 'taxa_aceitacao', 'discriminacao'}

# Métricas com casas decimais (as demais são exibidas como inteiros)
METRICAS_DECIMAIS = {'taxa_acerto', 'taxa_aceitacao', 'discriminacao'}

def formatar_numero(valor):
    """Formata número com separador de milhares (.) e sem decimais"""
    if pd.isna(valor):
        return 'N/A'
    try:
        v = float(valor)
        if v >= 1000 or v <= -1000:
            return f"{int(round(v)):,}".replace(',', '.')
        return str(int(round(v)))
    except:
        return 'N/A'

def formatar_decimal(valor, casas=3):
    """Formata decimal com vírgula como separador"""
    if pd.isna(valor):
        return 'N/A'
    try:
        formato = f"{{:.{casas}f}}"
        return formato.format(float(valor)).replace('.', ',')
    except:
        return 'N/A'

def formatar_metrica(valor, nome_coluna, casas=2):
    """Formata valor de acordo com o tipo da métrica"""
    if pd.isna(valor):
        return 'N/A'
    if nome_coluna in METRICAS_DECIMAIS:
        return formatar_decimal(valor, casas)
    else:
        return formatar_numero(valor)

# Configuração de estilo para gráficos
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("=" * 80)
print("ANÁLISE DAS MÉTRICAS DE DIFICULDADE")
print("=" * 80)

# 1. CARREGAR DADOS
print("\n1. Carregando dados...")
df = pd.read_csv(QUESTIONS_PATH)
print(f"   Total de questões no arquivo: {formatar_numero(len(df))}")

# 2. IDENTIFICAR MÉTRICAS DE DIFICULDADE DISPONÍVEIS NO CSV
# Apenas as colunas que existem no arquivo E estão no dicionário de tradução
difficulty_metrics = [col for col in traducao_metricas.keys() if col in df.columns]

print(f"\n2. Métricas de dificuldade identificadas: {len(difficulty_metrics)}")
for i, metric in enumerate(difficulty_metrics, 1):
    print(f"   MD{i}: {metric} → {traducao_metricas[metric]}")

# 3. FILTRAR QUESTÕES COM 16+ RESPOSTAS
print("\n3. Filtrando questões com 16 ou mais respostas...")
pc3_columns = [col for col in list(PC3_MAP.keys()) if col in df.columns]
df['total_responses'] = df[pc3_columns].sum(axis=1)
df_filtered = df[df['total_responses'] >= 16].copy()
print(f"   Questões com 16+ respostas: {formatar_numero(len(df_filtered))}")
print(f"   Questões excluídas: {formatar_numero(len(df) - len(df_filtered))}")

# Converter métricas para numérico
for metric in difficulty_metrics:
    df_filtered[metric] = pd.to_numeric(df_filtered[metric], errors='coerce')

# 4. CALCULAR ESTATÍSTICAS DESCRITIVAS
print("\n4. Calculando estatísticas descritivas...")
print("\n" + "=" * 80)
print("TABELA: Medidas de tendência central e dispersão das métricas de dificuldade")
print("Questões com 16 ou mais respostas")
print("=" * 80)

stats_list = []
valid_metrics = []

for metric in difficulty_metrics:
    values = df_filtered[metric].dropna()
    if len(values) > 0:
        stats = {
            'Métrica': traducao_metricas.get(metric, metric),
            'n': formatar_numero(len(values)),
            'Média': formatar_metrica(values.mean(), metric, casas=3),
            'Desvio-padrão': formatar_metrica(values.std(), metric, casas=3),
            'Min': formatar_metrica(values.min(), metric, casas=3),
            'Mediana': formatar_metrica(values.median(), metric, casas=3),
            'Max': formatar_metrica(values.max(), metric, casas=3),
        }
        stats_list.append(stats)
        valid_metrics.append(metric)
    else:
        print(f"   ⚠️  Métrica '{metric}' sem valores válidos. Ignorando.")

stats_df = pd.DataFrame(stats_list)
print(stats_df.to_string(index=False))

difficulty_metrics = valid_metrics

# 5. PLOTAR BOXPLOTS - DADOS ORIGINAIS
print("\n5. Gerando boxplots com dados originais...")

n_metrics = len(difficulty_metrics)
n_cols = 3
n_rows = int(np.ceil(n_metrics / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4.8 * n_rows))
plt.subplots_adjust(top=0.92, hspace=0.6, bottom=0.08)

if n_rows == 1:
    axes = axes.reshape(1, -1)
axes_flat = axes.flatten()

for idx, metric in enumerate(difficulty_metrics):
    ax = axes_flat[idx]
    data = df_filtered[metric].dropna()

    bp = ax.boxplot([data], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5))

    ax.set_xlim(0.5, 2.5)
    ax.set_title(f'MD{idx+1}: {traducao_metricas.get(metric, metric)}', fontweight='bold')
    ax.set_ylabel('valor')
    ax.set_xticklabels([''])
    ax.grid(True, alpha=0.3)

    # Formatar eixo Y de acordo com o tipo da métrica
    if metric in METRICAS_DECIMAIS:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_decimal(x, 2)))
    else:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_numero(x)))

    # Texto com estatísticas
    media_fmt   = formatar_metrica(data.mean(),   metric, casas=2)
    mediana_fmt = formatar_metrica(data.median(), metric, casas=2)
    std_fmt     = formatar_metrica(data.std(),    metric, casas=2)
    stats_text = f'média: {media_fmt}\nmediana: {mediana_fmt}\ndesvio padrão: {std_fmt}'

    ax.text(0.95, 0.5, stats_text, transform=ax.transAxes,
            verticalalignment='center', horizontalalignment='right',
            bbox=dict(boxstyle='round,pad=0.8', facecolor='wheat', alpha=0.6),
            fontsize=11, fontweight='bold')

for idx in range(n_metrics, len(axes_flat)):
    fig.delaxes(axes_flat[idx])

plt.tight_layout()
plt.show()
print("✓ Boxplots com dados originais exibidos")

# 5b. PLOTAR BOXPLOTS - Z-SCORE (MESMA ESCALA NO EIXO Y)
print("\n5b. Gerando boxplots com z-score (dados normalizados, mesma escala)...")

# Calcular todos os z-scores primeiro para determinar escala global
all_zscores = []
zscores_por_metrica = {}
for metric in difficulty_metrics:
    data = df_filtered[metric].dropna()
    std = data.std()
    z = (data - data.mean()) / std if std != 0 else data * 0
    zscores_por_metrica[metric] = z
    all_zscores.append(z)

# Determinar min e max globais com margem de 5%
z_global_min = min(z.min() for z in all_zscores)
z_global_max = max(z.max() for z in all_zscores)
margem = (z_global_max - z_global_min) * 0.05
y_min = z_global_min - margem
y_max = z_global_max + margem

print(f"   Escala global do eixo Y: [{formatar_decimal(y_min, 2)}, {formatar_decimal(y_max, 2)}]")

fig2, axes2 = plt.subplots(n_rows, n_cols, figsize=(15, 4.8 * n_rows))
plt.subplots_adjust(top=0.92, hspace=0.6, bottom=0.08)

if n_rows == 1:
    axes2 = axes2.reshape(1, -1)
axes2_flat = axes2.flatten()

def format_zscore(x, p):
    return str(int(x)) if x == int(x) else formatar_decimal(x, 1)

for idx, metric in enumerate(difficulty_metrics):
    ax = axes2_flat[idx]
    z_score = zscores_por_metrica[metric]

    bp = ax.boxplot([z_score], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightcoral', alpha=0.7),
                    medianprops=dict(color='darkred', linewidth=2),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5))

    ax.set_xlim(0.5, 2.5)
    ax.set_ylim(y_min, y_max)   # ← mesma escala em todos os subplots
    ax.set_title(f'MD{idx+1}: {traducao_metricas.get(metric, metric)}', fontweight='bold')
    ax.set_ylabel('z-score')
    ax.set_xticklabels([''])
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(FuncFormatter(format_zscore))
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)

    stats_text = f'mediana: {formatar_decimal(z_score.median(), 2)}'
    ax.text(0.95, 0.5, stats_text, transform=ax.transAxes,
            verticalalignment='center', horizontalalignment='right',
            bbox=dict(boxstyle='round,pad=0.8', facecolor='lightcoral', alpha=0.6),
            fontsize=11, fontweight='bold')

for idx in range(n_metrics, len(axes2_flat)):
    fig2.delaxes(axes2_flat[idx])

plt.tight_layout()
plt.show()
print("✓ Boxplots com z-score exibidos (escala unificada)")

# 6. CORRELAÇÃO DE SPEARMAN
print("\n6. Calculando correlação de Spearman entre métricas...")

df_metrics = df_filtered[difficulty_metrics].copy()
corr_df = df_metrics.corr(method='spearman')
corr_matrix = corr_df.values
n_metrics = len(difficulty_metrics)
translated_metrics = [traducao_metricas.get(m, m) for m in difficulty_metrics]

# 7. MATRIZ EM ESCADA (TEXTO)
print("\n" + "=" * 80)
print("MATRIZ DE CORRELAÇÃO DE SPEARMAN (forma de escada - TEXTO)")
print("=" * 80)
print()

header = f"{'Métrica':<25}"
for metric in translated_metrics:
    header += f"{metric[:12]:>14}"
print(header)
print("-" * (25 + 14 * n_metrics))

for i, metric_i in enumerate(translated_metrics):
    line = f"{metric_i:<25}"
    for j in range(n_metrics):
        if j <= i:
            val = corr_matrix[i, j]
            line += f"{formatar_decimal(val, 2) if not np.isnan(val) else 'N/A':>14}"
        else:
            line += f"{'':>14}"
    print(line)

# 8. HEATMAP TRIANGULAR
print("\n7. Gerando heatmap da matriz de correlação...")

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
annot_data = corr_df.applymap(lambda x: formatar_decimal(x, 2))

fig, ax = plt.subplots(figsize=(14, 12))
heatmap = sns.heatmap(corr_df,
            mask=mask,
            annot=annot_data,
            fmt='',
            cmap='coolwarm_r',
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8, "label": "Correlação de Spearman"},
            xticklabels=translated_metrics,
            yticklabels=translated_metrics,
            vmin=-1, vmax=1,
            ax=ax,
            annot_kws={'fontsize': 10})

colorbar = heatmap.collections[0].colorbar
colorbar.ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_decimal(x, 2)))

ax.set_xlabel('Métricas de dificuldade', fontsize=12, fontweight='bold')
ax.set_ylabel('Métricas de dificuldade', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()
print("✓ Heatmap exibido")

# 9. CORRELAÇÕES FORTES
print("\n8. Análise de correlações fortes...")
print("\n" + "=" * 80)
print("PARES DE MÉTRICAS COM CORRELAÇÃO FORTE (|ρ| > 0,7)")
print("=" * 80)

strong_corr = []
for i in range(n_metrics):
    for j in range(i):
        val = corr_matrix[i, j]
        if not np.isnan(val) and abs(val) > 0.7:
            strong_corr.append({
                'Métrica 1': traducao_metricas.get(difficulty_metrics[i], difficulty_metrics[i]),
                'Métrica 2': traducao_metricas.get(difficulty_metrics[j], difficulty_metrics[j]),
                'ρ': formatar_decimal(val, 2),
            })

if strong_corr:
    print(pd.DataFrame(strong_corr).to_string(index=False))
else:
    print("Nenhuma correlação forte encontrada (|ρ| > 0,7)")

# 10. CORRELAÇÕES MODERADAS
print("\n" + "=" * 80)
print("PARES DE MÉTRICAS COM CORRELAÇÃO MODERADA (0,5 < |ρ| ≤ 0,7)")
print("=" * 80)

moderate_corr = []
for i in range(n_metrics):
    for j in range(i):
        val = corr_matrix[i, j]
        if not np.isnan(val) and 0.5 < abs(val) <= 0.7:
            moderate_corr.append({
                'Métrica 1': traducao_metricas.get(difficulty_metrics[i], difficulty_metrics[i]),
                'Métrica 2': traducao_metricas.get(difficulty_metrics[j], difficulty_metrics[j]),
                'ρ': formatar_decimal(val, 2),
            })

if moderate_corr:
    print(pd.DataFrame(moderate_corr).to_string(index=False))
else:
    print("Nenhuma correlação moderada encontrada")

# 11. RESUMO FINAL
print("\n" + "=" * 80)
print("RESUMO DA ANÁLISE")
print("=" * 80)
print(f"✓ Total de questões analisadas: {formatar_numero(len(df_filtered))}")
print(f"✓ Número de métricas de dificuldade válidas: {len(difficulty_metrics)}")
print(f"✓ Métricas analisadas:")
for i, metric in enumerate(difficulty_metrics, 1):
    print(f"   MD{i}: {metric} → {traducao_metricas[metric]}")
print(f"\n⚠️  Métrica 'dificuldade' ignorada conforme configuração.")
print("\n" + "=" * 80)
print("ANÁLISE CONCLUÍDA COM SUCESSO!")
print("=" * 80)

# 3 - Versão PDF

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ── Texto selecionável no PDF ──────────────────────────────
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42
# ──────────────────────────────────────────────────────────

# Configurações
QUESTIONS_PATH = "output/dataset_analise_questoes.csv"

# Configurações de fonte global
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Mapeamento de Colunas para Nomes Legíveis
PC3_MAP = {
    'A4': 'A4', 'A5': 'A5', 'B4': 'B4', 'B6': 'B6',
    'B8': 'B8', 'B9': 'B9', 'B12': 'B12', 'C1': 'C1',
    'C2': 'C2', 'C3': 'C3', 'C4': 'C4', 'C8': 'C8',
    'D4': 'D4', 'E2': 'E2', 'G4': 'G4', 'G5': 'G5',
    'H1': 'H1'
}

traducao_metricas = {
    'taxa_acerto':           'taxa de acerto',
    'num_submissoes':        'submissões',
    'taxa_aceitacao':        'taxa de aceitação',
    'num_testes':            'testes',
    'num_consultas':         'consultas',
    'num_erros_lgcs':        'erros lógicos',
    'num_errors_stx':        'erros sintaxe',
    'num_erros':             'erros totais',
    'num_eventos':           'eventos totais',
    'num_eventos_del':       'eventos deleção',
    'tempo_implementacao':   'tempo implementação',
    'qtd_alteracoes_codigo': 'alterações código',
    'discriminacao':         'discriminação',
}

METRICAS_POSITIVAS = {'taxa_acerto', 'taxa_aceitacao', 'discriminacao'}
METRICAS_DECIMAIS  = {'taxa_acerto', 'taxa_aceitacao', 'discriminacao'}

def formatar_numero(valor):
    if pd.isna(valor):
        return 'N/A'
    try:
        v = float(valor)
        if v >= 1000 or v <= -1000:
            return f"{int(round(v)):,}".replace(',', '.')
        return str(int(round(v)))
    except:
        return 'N/A'

def formatar_decimal(valor, casas=3):
    if pd.isna(valor):
        return 'N/A'
    try:
        formato = f"{{:.{casas}f}}"
        return formato.format(float(valor)).replace('.', ',')
    except:
        return 'N/A'

def formatar_metrica(valor, nome_coluna, casas=2):
    if pd.isna(valor):
        return 'N/A'
    if nome_coluna in METRICAS_DECIMAIS:
        return formatar_decimal(valor, casas)
    else:
        return formatar_numero(valor)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("=" * 80)
print("ANÁLISE DAS MÉTRICAS DE DIFICULDADE")
print("=" * 80)

# 1. CARREGAR DADOS
print("\n1. Carregando dados...")
df = pd.read_csv(QUESTIONS_PATH)
print(f"   Total de questões no arquivo: {formatar_numero(len(df))}")

# 2. IDENTIFICAR MÉTRICAS
difficulty_metrics = [col for col in traducao_metricas.keys() if col in df.columns]
print(f"\n2. Métricas de dificuldade identificadas: {len(difficulty_metrics)}")
for i, metric in enumerate(difficulty_metrics, 1):
    print(f"   MD{i}: {metric} → {traducao_metricas[metric]}")

# 3. FILTRAR QUESTÕES COM 16+ RESPOSTAS
print("\n3. Filtrando questões com 16 ou mais respostas...")
pc3_columns = [col for col in list(PC3_MAP.keys()) if col in df.columns]
df['total_responses'] = df[pc3_columns].sum(axis=1)
df_filtered = df[df['total_responses'] >= 16].copy()
print(f"   Questões com 16+ respostas: {formatar_numero(len(df_filtered))}")
print(f"   Questões excluídas: {formatar_numero(len(df) - len(df_filtered))}")

for metric in difficulty_metrics:
    df_filtered[metric] = pd.to_numeric(df_filtered[metric], errors='coerce')

# 4. ESTATÍSTICAS DESCRITIVAS
print("\n4. Calculando estatísticas descritivas...")
stats_list = []
valid_metrics = []

for metric in difficulty_metrics:
    values = df_filtered[metric].dropna()
    if len(values) > 0:
        stats_list.append({
            'Métrica': traducao_metricas.get(metric, metric),
            'n': formatar_numero(len(values)),
            'Média': formatar_metrica(values.mean(),   metric, casas=3),
            'Desvio-padrão': formatar_metrica(values.std(),    metric, casas=3),
            'Min': formatar_metrica(values.min(),    metric, casas=3),
            'Mediana': formatar_metrica(values.median(), metric, casas=3),
            'Max': formatar_metrica(values.max(),    metric, casas=3),
        })
        valid_metrics.append(metric)
    else:
        print(f"   ⚠️  Métrica '{metric}' sem valores válidos. Ignorando.")

stats_df = pd.DataFrame(stats_list)
print(stats_df.to_string(index=False))
difficulty_metrics = valid_metrics

# 5. BOXPLOTS - DADOS ORIGINAIS → metricas_dificuldade_padrao.pdf
print("\n5. Gerando metricas_dificuldade_padrao.pdf ...")

n_metrics = len(difficulty_metrics)
n_cols = 3
n_rows = int(np.ceil(n_metrics / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4.8 * n_rows))
plt.subplots_adjust(top=0.92, hspace=0.6, bottom=0.08)

if n_rows == 1:
    axes = axes.reshape(1, -1)
axes_flat = axes.flatten()

for idx, metric in enumerate(difficulty_metrics):
    ax = axes_flat[idx]
    data = df_filtered[metric].dropna()

    bp = ax.boxplot([data], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5))

    ax.set_xlim(0.5, 2.5)
    ax.set_title(f'MD{idx+1}: {traducao_metricas.get(metric, metric)}', fontweight='bold')
    ax.set_ylabel('valor')
    ax.set_xticklabels([''])
    ax.grid(True, alpha=0.3)

    if metric in METRICAS_DECIMAIS:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_decimal(x, 2)))
    else:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_numero(x)))

    media_fmt   = formatar_metrica(data.mean(),   metric, casas=2)
    mediana_fmt = formatar_metrica(data.median(), metric, casas=2)
    std_fmt     = formatar_metrica(data.std(),    metric, casas=2)
    stats_text  = f'média: {media_fmt}\nmediana: {mediana_fmt}\ndesvio padrão: {std_fmt}'

    ax.text(0.95, 0.5, stats_text, transform=ax.transAxes,
            verticalalignment='center', horizontalalignment='right',
            bbox=dict(boxstyle='round,pad=0.8', facecolor='wheat', alpha=0.6),
            fontsize=11, fontweight='bold')

for idx in range(n_metrics, len(axes_flat)):
    fig.delaxes(axes_flat[idx])

plt.tight_layout()
plt.savefig("metricas_dificuldade_padrao.pdf", bbox_inches='tight', format='pdf')
plt.close()
print("💾 Salvo: metricas_dificuldade_padrao.pdf")

# 5b. BOXPLOTS - Z-SCORE → metricas_dificuldade_zscore.pdf
print("\n5b. Gerando metricas_dificuldade_zscore.pdf ...")

all_zscores = []
zscores_por_metrica = {}
for metric in difficulty_metrics:
    data = df_filtered[metric].dropna()
    std  = data.std()
    z    = (data - data.mean()) / std if std != 0 else data * 0
    zscores_por_metrica[metric] = z
    all_zscores.append(z)

z_global_min = min(z.min() for z in all_zscores)
z_global_max = max(z.max() for z in all_zscores)
margem = (z_global_max - z_global_min) * 0.05
y_min  = z_global_min - margem
y_max  = z_global_max + margem

print(f"   Escala global do eixo Y: [{formatar_decimal(y_min, 2)}, {formatar_decimal(y_max, 2)}]")

fig2, axes2 = plt.subplots(n_rows, n_cols, figsize=(15, 4.8 * n_rows))
plt.subplots_adjust(top=0.92, hspace=0.6, bottom=0.08)

if n_rows == 1:
    axes2 = axes2.reshape(1, -1)
axes2_flat = axes2.flatten()

def format_zscore(x, p):
    return str(int(x)) if x == int(x) else formatar_decimal(x, 1)

for idx, metric in enumerate(difficulty_metrics):
    ax      = axes2_flat[idx]
    z_score = zscores_por_metrica[metric]

    bp = ax.boxplot([z_score], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightcoral', alpha=0.7),
                    medianprops=dict(color='darkred', linewidth=2),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5))

    ax.set_xlim(0.5, 2.5)
    ax.set_ylim(y_min, y_max)
    ax.set_title(f'MD{idx+1}: {traducao_metricas.get(metric, metric)}', fontweight='bold')
    ax.set_ylabel('z-score')
    ax.set_xticklabels([''])
    ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(FuncFormatter(format_zscore))
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)

    stats_text = f'mediana: {formatar_decimal(z_score.median(), 2)}'
    ax.text(0.95, 0.5, stats_text, transform=ax.transAxes,
            verticalalignment='center', horizontalalignment='right',
            bbox=dict(boxstyle='round,pad=0.8', facecolor='lightcoral', alpha=0.6),
            fontsize=11, fontweight='bold')

for idx in range(n_metrics, len(axes2_flat)):
    fig2.delaxes(axes2_flat[idx])

plt.tight_layout()
plt.savefig("metricas_dificuldade_zscore.pdf", bbox_inches='tight', format='pdf')
plt.close()
print("💾 Salvo: metricas_dificuldade_zscore.pdf")

# 6. CORRELAÇÃO DE SPEARMAN
print("\n6. Calculando correlação de Spearman entre métricas...")

df_metrics        = df_filtered[difficulty_metrics].copy()
corr_df           = df_metrics.corr(method='spearman')
corr_matrix       = corr_df.values
n_metrics         = len(difficulty_metrics)
translated_metrics = [traducao_metricas.get(m, m) for m in difficulty_metrics]

# 7. MATRIZ EM ESCADA (TEXTO)
print("\n" + "=" * 80)
print("MATRIZ DE CORRELAÇÃO DE SPEARMAN (forma de escada - TEXTO)")
print("=" * 80)
print()

header = f"{'Métrica':<25}"
for metric in translated_metrics:
    header += f"{metric[:12]:>14}"
print(header)
print("-" * (25 + 14 * n_metrics))

for i, metric_i in enumerate(translated_metrics):
    line = f"{metric_i:<25}"
    for j in range(n_metrics):
        if j <= i:
            val = corr_matrix[i, j]
            line += f"{formatar_decimal(val, 2) if not np.isnan(val) else 'N/A':>14}"
        else:
            line += f"{'':>14}"
    print(line)

# 8. HEATMAP → correlacao_metricas_dificuldade.pdf
print("\n7. Gerando correlacao_metricas_dificuldade.pdf ...")

mask        = np.triu(np.ones_like(corr_matrix, dtype=bool))
annot_data  = corr_df.applymap(lambda x: formatar_decimal(x, 2))

fig, ax = plt.subplots(figsize=(10, 9))
heatmap = sns.heatmap(corr_df,
            mask=mask,
            annot=annot_data,
            fmt='',
            cmap='coolwarm_r',
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8, "label": "Correlação de Spearman"},
            xticklabels=translated_metrics,
            yticklabels=translated_metrics,
            vmin=-1, vmax=1,
            ax=ax,
            annot_kws={'fontsize': 10})

colorbar = heatmap.collections[0].colorbar
colorbar.ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_decimal(x, 2)))

ax.set_xlabel('Métricas de dificuldade', fontsize=12, fontweight='bold')
ax.set_ylabel('Métricas de dificuldade', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("correlacao_metricas_dificuldade.pdf", bbox_inches='tight', format='pdf')
plt.close()
print("💾 Salvo: correlacao_metricas_dificuldade.pdf")

# 9. CORRELAÇÕES FORTES
print("\n8. Análise de correlações fortes...")
print("\n" + "=" * 80)
print("PARES DE MÉTRICAS COM CORRELAÇÃO FORTE (|ρ| > 0,7)")
print("=" * 80)

strong_corr = []
for i in range(n_metrics):
    for j in range(i):
        val = corr_matrix[i, j]
        if not np.isnan(val) and abs(val) > 0.7:
            strong_corr.append({
                'Métrica 1': traducao_metricas.get(difficulty_metrics[i], difficulty_metrics[i]),
                'Métrica 2': traducao_metricas.get(difficulty_metrics[j], difficulty_metrics[j]),
                'ρ': formatar_decimal(val, 2),
            })

if strong_corr:
    print(pd.DataFrame(strong_corr).to_string(index=False))
else:
    print("Nenhuma correlação forte encontrada (|ρ| > 0,7)")

# 10. CORRELAÇÕES MODERADAS
print("\n" + "=" * 80)
print("PARES DE MÉTRICAS COM CORRELAÇÃO MODERADA (0,5 < |ρ| ≤ 0,7)")
print("=" * 80)

moderate_corr = []
for i in range(n_metrics):
    for j in range(i):
        val = corr_matrix[i, j]
        if not np.isnan(val) and 0.5 < abs(val) <= 0.7:
            moderate_corr.append({
                'Métrica 1': traducao_metricas.get(difficulty_metrics[i], difficulty_metrics[i]),
                'Métrica 2': traducao_metricas.get(difficulty_metrics[j], difficulty_metrics[j]),
                'ρ': formatar_decimal(val, 2),
            })

if moderate_corr:
    print(pd.DataFrame(moderate_corr).to_string(index=False))
else:
    print("Nenhuma correlação moderada encontrada")

# 11. RESUMO FINAL
print("\n" + "=" * 80)
print("RESUMO DA ANÁLISE")
print("=" * 80)
print(f"✓ Total de questões analisadas: {formatar_numero(len(df_filtered))}")
print(f"✓ Número de métricas de dificuldade válidas: {len(difficulty_metrics)}")
print(f"✓ Métricas analisadas:")
for i, metric in enumerate(difficulty_metrics, 1):
    print(f"   MD{i}: {metric} → {traducao_metricas[metric]}")
print(f"\n⚠️  Métrica 'dificuldade' ignorada conforme configuração.")
print("\n" + "=" * 80)
print("ANÁLISE CONCLUÍDA COM SUCESSO!")
print("=" * 80)

# 4 - Correlação entre PC³ e métricas de dificuldade

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# Configurações
QUESTIONS_PATH = "output/dataset_analise_questoes.csv"

# Mapeamento de Colunas PC³
PC3_MAP = {
    'A4': 'A4', 'A5': 'A5', 'B4': 'B4', 'B6': 'B6',
    'B8': 'B8', 'B9': 'B9', 'B12': 'B12', 'C1': 'C1',
    'C2': 'C2', 'C3': 'C3', 'C4': 'C4', 'C8': 'C8',
    'D4': 'D4', 'E2': 'E2', 'G4': 'G4', 'G5': 'G5',
    'H1': 'H1'
}

# ============================================================================
# MAPEAMENTO DAS MÉTRICAS DE DIFICULDADE
# Colunas reais do CSV → nomes legíveis
# 'dificuldade' é ignorada intencionalmente
# M1:  taxa_acerto           = num_correct / num_students_interactions
# M2:  num_submissoes        = num_submissions / num_students_interactions
# M3:  taxa_aceitacao        = num_correct / num_submissions
# M4:  num_testes            = num_tests / num_students_interactions
# M5:  num_consultas         = (num_submissions + num_tests) / num_students_interactions
# M6:  num_erros_lgcs        = num_logic_errors / num_students_interactions
# M7:  num_errors_stx        = num_syntax_errors / num_students_interactions
# M8:  num_erros             = num_errors / num_students_interactions
# M9:  num_eventos           = num_events / num_students_interactions
# M10: num_eventos_del       = num_deletes / num_students_interactions
# M11: tempo_implementacao   = code_time / num_correct
# M12: qtd_alteracoes_codigo = amount_of_change / num_students_interactions
# M13: discriminacao         (fórmula própria - já calculada)
# ============================================================================
traducao_metricas = {
    'taxa_acerto':           'taxa de acerto',      # M1
    'num_submissoes':        'submissões',           # M2
    'taxa_aceitacao':        'taxa de aceitação',    # M3
    'num_testes':            'testes',               # M4
    'num_consultas':         'consultas',            # M5
    'num_erros_lgcs':        'erros lógicos',        # M6
    'num_errors_stx':        'erros sintaxe',        # M7
    'num_erros':             'erros totais',         # M8
    'num_eventos':           'eventos totais',       # M9
    'num_eventos_del':       'eventos deleção',      # M10
    'tempo_implementacao':   'tempo implementação',  # M11
    'qtd_alteracoes_codigo': 'alterações código',    # M12
    'discriminacao':         'discriminação',        # M13
}

def formatar_decimal(valor, casas=2):
    """Formata decimal com vírgula"""
    if pd.isna(valor) or np.isnan(valor):
        return 'N/A'
    formato = f"{{:.{casas}f}}"
    return formato.format(valor).replace('.', ',')

# Configuração de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

def analisar_correlacao_pc3_metricas(top_n=10):
    """
    Analisa correlação entre PC³ e Métricas de Dificuldade

    Parâmetros:
        top_n (int): Número de PC³ mais frequentes a analisar. Default: 10
                     Use None para analisar todos os PC³ ativos.
    """

    print("=" * 80)
    print("ANÁLISE DE CORRELAÇÃO: PC³ vs MÉTRICAS DE DIFICULDADE")
    print("=" * 80)

    # 1. CARREGAR DADOS
    print("\n1. Carregando dados...")
    df = pd.read_csv(QUESTIONS_PATH)
    print(f"   Total de questões no arquivo: {len(df)}")

    # 2. FILTRAR QUESTÕES COM 16+ RESPOSTAS
    print("\n2. Filtrando questões com 16+ respostas...")
    pc3_columns = [col for col in list(PC3_MAP.keys()) if col in df.columns]
    df['total_responses'] = df[pc3_columns].sum(axis=1)
    df_filtered = df[df['total_responses'] >= 16].copy()
    print(f"   Questões com 16+ respostas: {len(df_filtered)}")
    print(f"   Questões excluídas: {len(df) - len(df_filtered)}")

    # 3. IDENTIFICAR COLUNAS PC³ ATIVAS (não vazias)
    print("\n3. Identificando PC³ ativos...")
    pc3_active = []
    pc3_removed = []

    for col in pc3_columns:
        if df_filtered[col].sum() > 0:
            pc3_active.append(col)
        else:
            pc3_removed.append(PC3_MAP[col])

    if pc3_removed:
        print(f"   ⚠️ PC³ removidos (sem ocorrências): {', '.join(pc3_removed)}")
    print(f"   ✓ PC³ ativos totais: {len(pc3_active)}")

    # 4. IDENTIFICAR MÉTRICAS DE DIFICULDADE DISPONÍVEIS NO CSV
    print("\n4. Identificando métricas de dificuldade válidas...")
    difficulty_metrics = []

    for metric in traducao_metricas.keys():
        if metric in df_filtered.columns:
            try:
                df_filtered[metric] = pd.to_numeric(df_filtered[metric], errors='coerce')
                if df_filtered[metric].dropna().shape[0] > 0:
                    difficulty_metrics.append(metric)
            except:
                pass
        else:
            print(f"   ⚠️ Coluna '{metric}' não encontrada no CSV. Ignorando.")

    print(f"   ✓ Métricas válidas para análise: {len(difficulty_metrics)}")
    for i, m in enumerate(difficulty_metrics, 1):
        print(f"     MD{i}: {m} → {traducao_metricas[m]}")

    # 5. ORDENAR PC³ POR FREQUÊNCIA E APLICAR TOP_N
    print("\n5. Ordenando PC³ por frequência...")
    pc3_totals = {col: df_filtered[col].sum() for col in pc3_active}
    pc3_sorted = sorted(pc3_totals.items(), key=lambda x: x[1], reverse=True)

    if top_n is not None and top_n < len(pc3_sorted):
        pc3_sorted = pc3_sorted[:top_n]
        print(f"   ℹ️ Selecionando os Top {top_n} PC³ mais frequentes")
    else:
        print(f"   ℹ️ Analisando todos os {len(pc3_sorted)} PC³ ativos")

    pc3_ordered = [col for col, _ in pc3_sorted]

    print("   Ordem (decrescente por frequência):")
    display_limit = min(10, len(pc3_ordered))
    for i, col in enumerate(pc3_ordered[:display_limit], 1):
        print(f"      {i}. {PC3_MAP[col]}: {int(pc3_totals[col])} ocorrências")
    if len(pc3_ordered) > display_limit:
        print(f"      ... e mais {len(pc3_ordered) - display_limit} PC³")

    # 6. CALCULAR CORRELAÇÃO ENTRE PC³ E MÉTRICAS
    print("\n6. Calculando correlação de Spearman (PC³ vs Métricas)...")

    n_pc3 = len(pc3_ordered)
    n_metrics = len(difficulty_metrics)

    corr_cross = np.zeros((n_pc3, n_metrics))

    for i, pc3_col in enumerate(pc3_ordered):
        for j, metric_col in enumerate(difficulty_metrics):
            valid_mask = df_filtered[pc3_col].notna() & df_filtered[metric_col].notna()
            if valid_mask.sum() > 1:
                corr_val = df_filtered.loc[valid_mask, [pc3_col, metric_col]].corr(method='spearman').iloc[0, 1]
                corr_cross[i, j] = corr_val
            else:
                corr_cross[i, j] = np.nan

    pc3_names = [PC3_MAP[col] for col in pc3_ordered]
    metric_names = [traducao_metricas[m] for m in difficulty_metrics]

    # 7. EXIBIR MATRIZ (TEXTO)
    print("\n" + "=" * 80)
    print("MATRIZ DE CORRELAÇÃO: PC³ (linhas) vs MÉTRICAS (colunas)")
    print("=" * 80)
    print()

    header = f"{'PC³':<20}"
    for name in metric_names:
        header += f"{name[:12]:>14}"
    print(header)
    print("-" * (20 + 14 * n_metrics))

    for i, pc3_name in enumerate(pc3_names):
        line = f"{pc3_name:<20}"
        for j in range(n_metrics):
            val = corr_cross[i, j]
            line += f"{'N/A':>14}" if np.isnan(val) else f"{formatar_decimal(val, 2):>14}"
        print(line)

    # 8. HEATMAP COMPLETO
    print("\n7. Gerando heatmap...")

    plt.figure(figsize=(n_metrics * 0.9, n_pc3 * 1.5))

    corr_df = pd.DataFrame(corr_cross, index=pc3_names, columns=metric_names)
    annot_data = corr_df.applymap(lambda x: formatar_decimal(x, 2))

    sns.heatmap(corr_df,
                annot=annot_data,
                fmt='',
                cmap='coolwarm_r',
                center=0,
                vmin=-1,
                vmax=1,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8, "label": "Correlação de Spearman"})

    plt.title('\n', fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Métricas de dificuldade', fontsize=12, fontweight='bold')
    plt.ylabel('PC³', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)

    ax = plt.gca()
    colorbar = ax.collections[0].colorbar
    colorbar.ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_decimal(x, 2)))

    plt.tight_layout()
    plt.show()
    print("✓ Heatmap exibido")

    # 9. CORRELAÇÕES FORTES
    print("\n8. Analisando correlações PC³ ↔ Métricas...")
    print("\n" + "=" * 80)
    print("CORRELAÇÕES FORTES ENTRE PC³ E MÉTRICAS (|ρ| > 0,7)")
    print("=" * 80)

    strong_cross = []
    for i in range(n_pc3):
        for j in range(n_metrics):
            val = corr_cross[i, j]
            if not np.isnan(val) and abs(val) > 0.7:
                strong_cross.append({
                    'PC³': pc3_names[i],
                    'Métrica': metric_names[j],
                    'ρ': formatar_decimal(val, 2)
                })

    if strong_cross:
        print(pd.DataFrame(strong_cross).to_string(index=False))
    else:
        print("Nenhuma correlação forte encontrada entre PC³ e Métricas.")

    # 10. CORRELAÇÕES MODERADAS
    print("\n" + "=" * 80)
    print("CORRELAÇÕES MODERADAS ENTRE PC³ E MÉTRICAS (0,5 < |ρ| ≤ 0,7)")
    print("=" * 80)

    moderate_cross = []
    for i in range(n_pc3):
        for j in range(n_metrics):
            val = corr_cross[i, j]
            if not np.isnan(val) and 0.5 < abs(val) <= 0.7:
                moderate_cross.append({
                    'PC³': pc3_names[i],
                    'Métrica': metric_names[j],
                    'ρ': formatar_decimal(val, 2)
                })

    if moderate_cross:
        print(pd.DataFrame(moderate_cross).to_string(index=False))
    else:
        print("Nenhuma correlação moderada encontrada.")

    # 11. RESUMO FINAL
    print("\n" + "=" * 80)
    print("RESUMO DA ANÁLISE")
    print("=" * 80)
    print(f"✓ Total de questões analisadas: {len(df_filtered)}")
    print(f"✓ PC³ analisados: {n_pc3} {'(Top ' + str(top_n) + ')' if top_n else '(Todos)'}")
    print(f"✓ Métricas de dificuldade analisadas: {n_metrics}")
    print(f"✓ Dimensão da matriz: {n_pc3} PC³ × {n_metrics} Métricas")
    print(f"✓ Correlações fortes  (|ρ| > 0,7): {len(strong_cross)}")
    print(f"✓ Correlações moderadas (0,5 < |ρ| ≤ 0,7): {len(moderate_cross)}")
    print(f"\n⚠️  Métrica 'dificuldade' ignorada conforme configuração.")
    print("\n" + "=" * 80)
    print("ANÁLISE CONCLUÍDA COM SUCESSO!")
    print("=" * 80)


# EXECUTAR ANÁLISE
print("\n")
print("💡 USO: analisar_correlacao_pc3_metricas(top_n=10)")
print("   - top_n=10   : Analisa os 10 PC³ mais frequentes (padrão)")
print("   - top_n=None : Analisa todos os PC³ ativos")
print("=" * 80)
print("\n")

analisar_correlacao_pc3_metricas(top_n=4)

# 4 - Versão PDF

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ── Texto selecionável no PDF ──────────────────────────────
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42
# ──────────────────────────────────────────────────────────

# Configurações
QUESTIONS_PATH = "output/dataset_analise_questoes.csv"

# Mapeamento de Colunas PC³
PC3_MAP = {
    'A4': 'A4', 'A5': 'A5', 'B4': 'B4', 'B6': 'B6',
    'B8': 'B8', 'B9': 'B9', 'B12': 'B12', 'C1': 'C1',
    'C2': 'C2', 'C3': 'C3', 'C4': 'C4', 'C8': 'C8',
    'D4': 'D4', 'E2': 'E2', 'G4': 'G4', 'G5': 'G5',
    'H1': 'H1'
}

traducao_metricas = {
    'taxa_acerto':           'taxa de acerto',
    'num_submissoes':        'submissões',
    'taxa_aceitacao':        'taxa de aceitação',
    'num_testes':            'testes',
    'num_consultas':         'consultas',
    'num_erros_lgcs':        'erros lógicos',
    'num_errors_stx':        'erros sintaxe',
    'num_erros':             'erros totais',
    'num_eventos':           'eventos totais',
    'num_eventos_del':       'eventos deleção',
    'tempo_implementacao':   'tempo implementação',
    'qtd_alteracoes_codigo': 'alterações código',
    'discriminacao':         'discriminação',
}

def formatar_decimal(valor, casas=2):
    if pd.isna(valor) or np.isnan(valor):
        return 'N/A'
    formato = f"{{:.{casas}f}}"
    return formato.format(valor).replace('.', ',')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

def analisar_correlacao_pc3_metricas(top_n=10):
    print("=" * 80)
    print("ANÁLISE DE CORRELAÇÃO: PC³ vs MÉTRICAS DE DIFICULDADE")
    print("=" * 80)

    # 1. CARREGAR DADOS
    print("\n1. Carregando dados...")
    df = pd.read_csv(QUESTIONS_PATH)
    print(f"   Total de questões no arquivo: {len(df)}")

    # 2. FILTRAR QUESTÕES COM 16+ RESPOSTAS
    print("\n2. Filtrando questões com 16+ respostas...")
    pc3_columns = [col for col in list(PC3_MAP.keys()) if col in df.columns]
    df['total_responses'] = df[pc3_columns].sum(axis=1)
    df_filtered = df[df['total_responses'] >= 16].copy()
    print(f"   Questões com 16+ respostas: {len(df_filtered)}")
    print(f"   Questões excluídas: {len(df) - len(df_filtered)}")

    # 3. IDENTIFICAR PC³ ATIVOS
    print("\n3. Identificando PC³ ativos...")
    pc3_active  = []
    pc3_removed = []

    for col in pc3_columns:
        if df_filtered[col].sum() > 0:
            pc3_active.append(col)
        else:
            pc3_removed.append(PC3_MAP[col])

    if pc3_removed:
        print(f"   ⚠️ PC³ removidos (sem ocorrências): {', '.join(pc3_removed)}")
    print(f"   ✓ PC³ ativos totais: {len(pc3_active)}")

    # 4. IDENTIFICAR MÉTRICAS VÁLIDAS
    print("\n4. Identificando métricas de dificuldade válidas...")
    difficulty_metrics = []

    for metric in traducao_metricas.keys():
        if metric in df_filtered.columns:
            try:
                df_filtered[metric] = pd.to_numeric(df_filtered[metric], errors='coerce')
                if df_filtered[metric].dropna().shape[0] > 0:
                    difficulty_metrics.append(metric)
            except:
                pass
        else:
            print(f"   ⚠️ Coluna '{metric}' não encontrada no CSV. Ignorando.")

    print(f"   ✓ Métricas válidas para análise: {len(difficulty_metrics)}")
    for i, m in enumerate(difficulty_metrics, 1):
        print(f"     MD{i}: {m} → {traducao_metricas[m]}")

    # 5. ORDENAR PC³ POR FREQUÊNCIA E APLICAR TOP_N
    print("\n5. Ordenando PC³ por frequência...")
    pc3_totals = {col: df_filtered[col].sum() for col in pc3_active}
    pc3_sorted = sorted(pc3_totals.items(), key=lambda x: x[1], reverse=True)

    if top_n is not None and top_n < len(pc3_sorted):
        pc3_sorted = pc3_sorted[:top_n]
        print(f"   ℹ️ Selecionando os Top {top_n} PC³ mais frequentes")
    else:
        print(f"   ℹ️ Analisando todos os {len(pc3_sorted)} PC³ ativos")

    pc3_ordered = [col for col, _ in pc3_sorted]

    print("   Ordem (decrescente por frequência):")
    display_limit = min(10, len(pc3_ordered))
    for i, col in enumerate(pc3_ordered[:display_limit], 1):
        print(f"      {i}. {PC3_MAP[col]}: {int(pc3_totals[col])} ocorrências")
    if len(pc3_ordered) > display_limit:
        print(f"      ... e mais {len(pc3_ordered) - display_limit} PC³")

    # 6. CALCULAR CORRELAÇÃO
    print("\n6. Calculando correlação de Spearman (PC³ vs Métricas)...")

    n_pc3     = len(pc3_ordered)
    n_metrics = len(difficulty_metrics)
    corr_cross = np.zeros((n_pc3, n_metrics))

    for i, pc3_col in enumerate(pc3_ordered):
        for j, metric_col in enumerate(difficulty_metrics):
            valid_mask = df_filtered[pc3_col].notna() & df_filtered[metric_col].notna()
            if valid_mask.sum() > 1:
                corr_val = df_filtered.loc[valid_mask, [pc3_col, metric_col]].corr(method='spearman').iloc[0, 1]
                corr_cross[i, j] = corr_val
            else:
                corr_cross[i, j] = np.nan

    pc3_names    = [PC3_MAP[col] for col in pc3_ordered]
    metric_names = [traducao_metricas[m] for m in difficulty_metrics]

    # 7. EXIBIR MATRIZ (TEXTO)
    print("\n" + "=" * 80)
    print("MATRIZ DE CORRELAÇÃO: PC³ (linhas) vs MÉTRICAS (colunas)")
    print("=" * 80)
    print()

    header = f"{'PC³':<20}"
    for name in metric_names:
        header += f"{name[:12]:>14}"
    print(header)
    print("-" * (20 + 14 * n_metrics))

    for i, pc3_name in enumerate(pc3_names):
        line = f"{pc3_name:<20}"
        for j in range(n_metrics):
            val = corr_cross[i, j]
            line += f"{'N/A':>14}" if np.isnan(val) else f"{formatar_decimal(val, 2):>14}"
        print(line)

    # 8. HEATMAP → correlacao_pc3_metricas.pdf
    print("\n7. Gerando correlacao_pc3_metricas.pdf ...")

    corr_df    = pd.DataFrame(corr_cross, index=pc3_names, columns=metric_names)
    annot_data = corr_df.applymap(lambda x: formatar_decimal(x, 2))

    fig, ax = plt.subplots(figsize=(n_metrics * 0.9, n_pc3 * 1.5))

    heatmap = sns.heatmap(corr_df,
                annot=annot_data,
                fmt='',
                cmap='coolwarm_r',
                center=0,
                vmin=-1,
                vmax=1,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8, "label": "Correlação de Spearman"},
                ax=ax)

    plt.title('\n', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Métricas de dificuldade', fontsize=12, fontweight='bold')
    ax.set_ylabel('PC³', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)

    colorbar = ax.collections[0].colorbar
    colorbar.ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: formatar_decimal(x, 2)))

    plt.tight_layout()
    plt.savefig("correlacao_metricas_dificuldade_pc3.pdf", bbox_inches='tight', format='pdf')
    plt.close()
    print("💾 Salvo: correlacao_metricas_dificuldade_pc3.pdf")

    # 9. CORRELAÇÕES FORTES
    print("\n8. Analisando correlações PC³ ↔ Métricas...")
    print("\n" + "=" * 80)
    print("CORRELAÇÕES FORTES ENTRE PC³ E MÉTRICAS (|ρ| > 0,7)")
    print("=" * 80)

    strong_cross = []
    for i in range(n_pc3):
        for j in range(n_metrics):
            val = corr_cross[i, j]
            if not np.isnan(val) and abs(val) > 0.7:
                strong_cross.append({
                    'PC³': pc3_names[i],
                    'Métrica': metric_names[j],
                    'ρ': formatar_decimal(val, 2)
                })

    if strong_cross:
        print(pd.DataFrame(strong_cross).to_string(index=False))
    else:
        print("Nenhuma correlação forte encontrada entre PC³ e Métricas.")

    # 10. CORRELAÇÕES MODERADAS
    print("\n" + "=" * 80)
    print("CORRELAÇÕES MODERADAS ENTRE PC³ E MÉTRICAS (0,5 < |ρ| ≤ 0,7)")
    print("=" * 80)

    moderate_cross = []
    for i in range(n_pc3):
        for j in range(n_metrics):
            val = corr_cross[i, j]
            if not np.isnan(val) and 0.5 < abs(val) <= 0.7:
                moderate_cross.append({
                    'PC³': pc3_names[i],
                    'Métrica': metric_names[j],
                    'ρ': formatar_decimal(val, 2)
                })

    if moderate_cross:
        print(pd.DataFrame(moderate_cross).to_string(index=False))
    else:
        print("Nenhuma correlação moderada encontrada.")

    # 11. RESUMO FINAL
    print("\n" + "=" * 80)
    print("RESUMO DA ANÁLISE")
    print("=" * 80)
    print(f"✓ Total de questões analisadas: {len(df_filtered)}")
    print(f"✓ PC³ analisados: {n_pc3} {'(Top ' + str(top_n) + ')' if top_n else '(Todos)'}")
    print(f"✓ Métricas de dificuldade analisadas: {n_metrics}")
    print(f"✓ Dimensão da matriz: {n_pc3} PC³ × {n_metrics} Métricas")
    print(f"✓ Correlações fortes  (|ρ| > 0,7): {len(strong_cross)}")
    print(f"✓ Correlações moderadas (0,5 < |ρ| ≤ 0,7): {len(moderate_cross)}")
    print(f"\n⚠️  Métrica 'dificuldade' ignorada conforme configuração.")
    print("\n" + "=" * 80)
    print("ANÁLISE CONCLUÍDA COM SUCESSO!")
    print("=" * 80)


# EXECUTAR ANÁLISE
print("\n")
print("💡 USO: analisar_correlacao_pc3_metricas(top_n=10)")
print("   - top_n=10   : Analisa os 10 PC³ mais frequentes (padrão)")
print("   - top_n=None : Analisa todos os PC³ ativos")
print("=" * 80)
print("\n")

analisar_correlacao_pc3_metricas(top_n=4)

# 5 - Comparação entre PC³ e métricas de dificuldade

In [ ]:
# Heatmap de correlação Spearman
def plotar_heatmap(df, metrics):
    corr = df[metrics].corr(method="spearman")
    plt.figure(figsize=(12,10))
    sns.heatmap(
        corr,
        annot=True, fmt=".2f",
        cmap="coolwarm",
        center=0,
        square=True,
        cbar_kws={"shrink": .8}
    )
    plt.title("Heatmap de correlação Spearman entre métricas de dificuldade")
    plt.tight_layout()
    plt.show()

def analisar_correlacao_pc3_metricas():
    df = pd.read_csv(QUESTIONS_PATH)
    df_filtered = df[df['respostas'] >= 16].copy()

    pc3_columns = [col for col in list(PC3_MAP.keys()) if col in df_filtered.columns]
    pc3_all = []
    for pc3 in pc3_columns:
        total_ocorrencias = df_filtered[pc3].sum()
        if total_ocorrencias > 0:
            pc3_all.append({
                'pc3_code': pc3,
                'PC3': PC3_MAP[pc3],
                'Total_Ocorrencias': int(total_ocorrencias),
            })
    pc3_all_sorted = sorted(pc3_all, key=lambda x: x['Total_Ocorrencias'], reverse=True)
    pc3_selected = pc3_all_sorted[:TOP_N_PC3]
    pc3_frequentes = [item['pc3_code'] for item in pc3_selected]

    difficulty_metrics = []
    for metric in traducao_metricas.keys():
        if metric in df_filtered.columns:
            df_filtered[metric] = pd.to_numeric(df_filtered[metric], errors='coerce')
            if df_filtered[metric].dropna().shape[0] > 5:
                difficulty_metrics.append(metric)

    for mi in pc3_frequentes:
        df_filtered[f'{mi}_pct'] = df_filtered[mi] / df_filtered['respostas']

    resultados = []
    for mi in pc3_frequentes:
        for metric in difficulty_metrics:
            dados = df_filtered[[f'{mi}_pct', metric]].dropna()
            n = len(dados)
            if n < 5:
                continue
            rho, p_value = spearmanr(dados[f'{mi}_pct'], dados[metric])
            resultados.append({
                'PC3': PC3_MAP[mi],
                'Metrica': traducao_metricas[metric],
                'metrica_original': metric,
                'pc3_code': mi,
                'rho': rho,
                'p_valor': p_value,
                'interpretacao': interpretar_rho(rho),
            })

    df_resultados = pd.DataFrame(resultados)
    df_resultados['p_valor_ajustado'] = benjamini_hochberg(df_resultados['p_valor'].values)
    df_resultados['significativo'] = df_resultados['p_valor_ajustado'] < ALPHA

    # Gráficos de dispersão
    for _, row in df_resultados[df_resultados['significativo']].iterrows():
        print(f"{row['PC3']} × {row['Metrica']} | ρ={row['rho']:.3f}, p_adj={row['p_valor_ajustado']:.3f} ({row['interpretacao']})")
        plotar_correlacao(df_filtered, row['pc3_code'], row['metrica_original'], traducao_metricas)

    # Gráfico de barras em escada
    plotar_barras_escada(df_resultados)

    # Heatmap de correlação entre métricas
    if difficulty_metrics:
        plotar_heatmap(df_filtered, difficulty_metrics)

    return df_resultados

# Executar análise
resultados = analisar_correlacao_pc3_metricas()


# 6 - Comparação entre PC³ e fatores demográficos

In [ ]:
import csv
from pathlib import Path

# ============================================================================
# CONFIGURAÇÕES
# ============================================================================

MISCONCEPTIONS_PATH = "output/misconceptions_detalhado_por_usuario.csv"
REFERENCES_PATH = Path('../Etapa_2/output/referencias_processamento.json')

# ============================================================================
# FUNÇÕES
# ============================================================================

def obter_lista_estudantes_catalogados():
    """
    Retorna LISTA de IDs de estudantes catalogados
    
    Similar a contar_estudantes(), mas retorna a LISTA em vez de apenas o count
    
    Retorna:
        set: conjunto de IDs únicos de usuários
    """
    estudantes = set()
    
    try:
        with open(MISCONCEPTIONS_PATH, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            
            for linha in reader:
                if 'usuario' in linha and linha['usuario']:
                    estudantes.add(linha['usuario'])
        
        return estudantes
    except FileNotFoundError:
        print(f"❌ Arquivo {MISCONCEPTIONS_PATH} não encontrado")
        return set()


# ============================================================================
# ANÁLISE PRINCIPAL
# ============================================================================

# 1. OBTER LISTA DE ESTUDANTES CATALOGADOS
print("=" * 70)
print("ANÁLISE DE user.data - APENAS ESTUDANTES CATALOGADOS")
print("=" * 70)

estudantes_catalogados = obter_lista_estudantes_catalogados()
print(f"\n📊 Estudantes catalogados: {len(estudantes_catalogados)}")

# 2. VERIFICAR TOTAL DE PASTAS NO DIRETÓRIO
caminho = REFERENCES_PATH

if not caminho.exists():
    print(f"❌ JSON de referências não encontrado: {caminho}")
    exit()

referencias = carregar_referencias(caminho)
total_pastas_no_disco = len(referencias.get('users', []))
print(f"📁 Total de usuários no índice: {total_pastas_no_disco}")

# 3. BUSCAR APENAS NAS PASTAS DOS ESTUDANTES CATALOGADOS
print(f"\n🔍 Buscando user.data APENAS dos {len(estudantes_catalogados)} estudantes catalogados...\n")

# Categorias para verificar
categorias = {
    'CURRENT DEGREE COURSE': 0,
    'HIGH SCHOOL': 0,
    'PERSONAL COMPUTER': 0,
    'WORK': 0,
    'PREVIOUS DEGREE': 0,
    'OTHER INFORMATION': 0
}

# Total de arquivos processados
total_arquivos_encontrados = 0
total_arquivos_nao_encontrados = 0

# ============================================================================
# ITERAR APENAS PELOS ESTUDANTES CATALOGADOS (não por todas as pastas!)
# ============================================================================

for usuario in estudantes_catalogados:
    # Construir caminho da pasta do usuário
    arquivos_user_data = list(iterar_arquivos_usuario(referencias, usuario, 'user_data', 'user.data'))
    arquivo_user_data = arquivos_user_data[0] if arquivos_user_data else None
    
    if arquivo_user_data is not None:
        total_arquivos_encontrados += 1
        
        # Ler o conteúdo do arquivo
        try:
            with open(arquivo_user_data, 'r', encoding='utf-8') as f:
                conteudo = f.read()
            
            # Verificar presença de cada categoria
            for categoria in categorias.keys():
                if f'-- {categoria}:' in conteudo:
                    categorias[categoria] += 1
                    
        except Exception as e:
            print(f"⚠️  Erro ao ler {arquivo_user_data}: {e}")
    else:
        total_arquivos_nao_encontrados += 1

# ============================================================================
# EXIBIR RESULTADOS
# ============================================================================

print("=" * 70)
print("RESULTADOS")
print("=" * 70)

print(f"\n📊 RESUMO:")
print(f"   • Total de pastas no disco: {total_pastas_no_disco}")
print(f"   • Estudantes catalogados: {len(estudantes_catalogados)}")
print(f"   • Arquivos user.data encontrados: {total_arquivos_encontrados}")
print(f"   • Arquivos user.data NÃO encontrados: {total_arquivos_nao_encontrados}")

if total_arquivos_nao_encontrados > 0:
    porcentagem_faltando = (total_arquivos_nao_encontrados / len(estudantes_catalogados)) * 100
    print(f"\n⚠️  {total_arquivos_nao_encontrados} estudantes catalogados ({porcentagem_faltando:.1f}%) não têm pasta em {caminho.name}")

print(f"\n📈 CONTAGEM POR CATEGORIA:")
print("-" * 70)

for categoria, count in categorias.items():
    if total_arquivos_encontrados > 0:
        porcentagem = (count / total_arquivos_encontrados * 100)
        print(f"-- {categoria:<25}: {count:>4} ({porcentagem:>5.1f}%)")
    else:
        print(f"-- {categoria:<25}: {count:>4}")

print("\n" + "=" * 70)

In [ ]:
import csv
from pathlib import Path
from collections import defaultdict

# ============================================================================
# CONFIGURAÇÕES
# ============================================================================

MISCONCEPTIONS_PATH = "output/misconceptions_detalhado_por_usuario.csv"
USER_DATA_PATH = Path('../Etapa_2/output/referencias_processamento.json')

# ============================================================================
# FUNÇÕES
# ============================================================================

def obter_lista_estudantes_catalogados():
    """Retorna set de IDs de estudantes catalogados"""
    estudantes = set()
    
    try:
        with open(MISCONCEPTIONS_PATH, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            
            for linha in reader:
                if 'usuario' in linha and linha['usuario']:
                    estudantes.add(linha['usuario'])
        
        return estudantes
    except FileNotFoundError:
        print(f"❌ Arquivo {MISCONCEPTIONS_PATH} não encontrado")
        return set()


def extrair_campos_categoria(conteudo, inicio_categoria, proxima_categoria=None):
    """
    Extrai todos os campos de uma categoria específica
    
    Retorna:
        dict: {campo: valor}
    """
    campos = {}
    
    # Encontrar onde começa a categoria
    if inicio_categoria not in conteudo:
        return campos
    
    inicio = conteudo.find(inicio_categoria)
    
    # Encontrar onde termina (próxima categoria ou fim do arquivo)
    if proxima_categoria:
        fim = conteudo.find(proxima_categoria, inicio)
        if fim == -1:
            fim = len(conteudo)
    else:
        fim = len(conteudo)
    
    # Extrair bloco da categoria
    bloco = conteudo[inicio:fim]
    linhas = bloco.split('\n')
    
    # Processar cada linha que começa com "----"
    for linha in linhas:
        linha = linha.strip()
        if linha.startswith('----'):
            # Remover "----" e espaços
            linha = linha.replace('----', '').strip()
            
            # Separar campo: valor
            if ':' in linha:
                campo, valor = linha.split(':', 1)
                campo = campo.strip()
                valor = valor.strip()
                
                # Se valor está vazio, marcar como "n/a"
                if not valor:
                    valor = "n/a"
                
                campos[campo] = valor
    
    return campos


def analisar_user_data_detalhado():
    """
    Análise DETALHADA de todas as subcategorias dos arquivos user.data
    """
    
    print("=" * 80)
    print("ANÁLISE DETALHADA DE user.data - ESTUDANTES CATALOGADOS")
    print("=" * 80)
    
    # 1. Obter estudantes catalogados
    estudantes_catalogados = obter_lista_estudantes_catalogados()
    print(f"\n📊 Estudantes catalogados: {len(estudantes_catalogados)}")
    
    # 2. Verificar caminho
    if not USER_DATA_PATH.exists():
        print(f"❌ JSON de referências não encontrado: {USER_DATA_PATH}")
        return

    referencias = carregar_referencias(USER_DATA_PATH)
    
    # 3. Estruturas para armazenar contagens
    # categorias[nome_categoria][campo][valor] = count
    categorias = {
        'CURRENT DEGREE COURSE': defaultdict(lambda: defaultdict(int)),
        'HIGH SCHOOL': defaultdict(lambda: defaultdict(int)),
        'PERSONAL COMPUTER': defaultdict(lambda: defaultdict(int)),
        'WORK': defaultdict(lambda: defaultdict(int)),
        'PREVIOUS DEGREE': defaultdict(lambda: defaultdict(int)),
        'OTHER INFORMATION': defaultdict(lambda: defaultdict(int))
    }
    
    total_arquivos_encontrados = 0
    
    # 4. Processar cada estudante catalogado
    print(f"\n🔍 Processando arquivos user.data...\n")
    
    categorias_ordem = [
        'CURRENT DEGREE COURSE',
        'HIGH SCHOOL',
        'PERSONAL COMPUTER',
        'WORK',
        'PREVIOUS DEGREE',
        'OTHER INFORMATION'
    ]
    
    for usuario in estudantes_catalogados:
        arquivos_user_data = list(iterar_arquivos_usuario(referencias, usuario, 'user_data', 'user.data'))
        arquivo_user_data = arquivos_user_data[0] if arquivos_user_data else None
        
        if arquivo_user_data is not None:
            total_arquivos_encontrados += 1
            
            try:
                with open(arquivo_user_data, 'r', encoding='utf-8') as f:
                    conteudo = f.read()
                
                # Processar cada categoria
                for i, categoria in enumerate(categorias_ordem):
                    # Definir próxima categoria (ou None se for a última)
                    proxima = categorias_ordem[i + 1] if i + 1 < len(categorias_ordem) else None
                    
                    # Extrair campos da categoria
                    campos = extrair_campos_categoria(
                        conteudo,
                        f'-- {categoria}:',
                        f'-- {proxima}:' if proxima else None
                    )
                    
                    # Contar cada campo:valor
                    for campo, valor in campos.items():
                        categorias[categoria][campo][valor] += 1
                
            except Exception as e:
                print(f"⚠️  Erro ao processar {arquivo_user_data}: {e}")
    
    # 5. EXIBIR RESULTADOS DETALHADOS
    print("=" * 80)
    print("RESULTADOS DETALHADOS POR CATEGORIA")
    print("=" * 80)
    print(f"\nTotal de arquivos processados: {total_arquivos_encontrados}\n")
    
    for categoria in categorias_ordem:
        campos = categorias[categoria]
        
        if not campos:
            continue
        
        print("\n" + "=" * 80)
        print(f"📋 {categoria}")
        print("=" * 80)
        
        for campo in sorted(campos.keys()):
            valores = campos[campo]
            total_respostas = sum(valores.values())
            
            print(f"\n   🔹 {campo}:")
            print(f"   {'─' * 70}")
            
            # Ordenar valores por frequência (decrescente)
            valores_ordenados = sorted(valores.items(), key=lambda x: x[1], reverse=True)
            
            for valor, count in valores_ordenados:
                porcentagem = (count / total_respostas) * 100
                
                # Truncar valores muito longos
                valor_exibido = valor if len(valor) <= 50 else valor[:47] + "..."
                
                print(f"      • {valor_exibido:<50} : {count:>4} ({porcentagem:>5.1f}%)")
    
    print("\n" + "=" * 80)
    print("ANÁLISE CONCLUÍDA!")
    print("=" * 80)


# ============================================================================
# EXECUTAR
# ============================================================================

if __name__ == "__main__":
    analisar_user_data_detalhado()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import csv
from pathlib import Path
from collections import defaultdict
from scipy.stats import chi2_contingency

MISCONCEPTIONS_PATH = "output/misconceptions_detalhado_por_usuario.csv"
REFERENCES_PATH = Path('../Etapa_2/output/referencias_processamento.json')
USER_DATA_PATH = REFERENCES_PATH

PC3_MAP = {
    'B4': 'B4', 'B8': 'B8', 'B9': 'B9', 'G4': 'G4'
}

def formatar_pvalor(p_val):
    if p_val < 1e-16: return '<< 0,001'
    elif p_val < 1e-3: return '< 0,001'
    else: return f"{p_val:.3f}".replace('.', ',')

def formatar_numero(num):
    if isinstance(num, (int, np.integer)):
        return f"{num:,}".replace(',', '.')
    else:
        num_str = f"{num:.3f}"
        partes = num_str.split('.')
        parte_inteira = f"{int(partes[0]):,}".replace(',', '.')
        parte_decimal = partes[1].rstrip('0') if len(partes) > 1 else ''
        return f"{parte_inteira},{parte_decimal}" if parte_decimal else parte_inteira

def extrair_campo_user_data(conteudo, campo_busca):
    linhas = conteudo.split('\n')
    for linha in linhas:
        linha = linha.strip()
        if linha.startswith('----'):
            linha_limpa = linha.replace('----', '').strip()
            if linha_limpa.lower().startswith(campo_busca.lower() + ':'):
                if ':' in linha_limpa:
                    _, valor = linha_limpa.split(':', 1)
                    valor = valor.strip()
                    return valor if valor else None
    return None

def carregar_dados_usuarios():
    usuarios_catalogados = set()
    colunas_padrao = ['usuario', 'school_type', 'previous_experience', 'share_pc', 'worked_interned']

    try:
        with open(MISCONCEPTIONS_PATH, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for linha in reader:
                if 'usuario' in linha and linha['usuario']:
                    usuarios_catalogados.add(linha['usuario'])
    except FileNotFoundError:
        print(f"⚠️  Arquivo {MISCONCEPTIONS_PATH} não encontrado!")
        return pd.DataFrame(columns=colunas_padrao)

    if not USER_DATA_PATH.exists():
        print(f"⚠️  JSON de referências {USER_DATA_PATH} não existe!")
        return pd.DataFrame(columns=colunas_padrao)
    manifest = carregar_referencias(USER_DATA_PATH)

    dados_usuarios = []
    for usuario in usuarios_catalogados:
        arquivos_user_data = list(iterar_arquivos_usuario(manifest, usuario, 'user_data', 'user.data'))
        arquivo_user_data = arquivos_user_data[0] if arquivos_user_data else None

        if arquivo_user_data is not None:
            try:
                with open(arquivo_user_data, 'r', encoding='utf-8') as f:
                    conteudo = f.read()

                school_type     = extrair_campo_user_data(conteudo, "school type")
                previous_exp    = extrair_campo_user_data(conteudo, "previous experience of any computer language")
                share_pc        = extrair_campo_user_data(conteudo, "share this PC with other people at home")
                worked_interned = extrair_campo_user_data(conteudo, "worked or interned before the degree")

                if previous_exp and previous_exp.lower().startswith('yes'):
                    previous_exp = 'yes'

                dados_usuarios.append({
                    'usuario': usuario,
                    'school_type': school_type,
                    'previous_experience': previous_exp,
                    'share_pc': share_pc,
                    'worked_interned': worked_interned
                })
            except Exception:
                pass

    if not dados_usuarios:
        print(f"⚠️  Nenhum dado de usuário foi coletado!")
        return pd.DataFrame(columns=colunas_padrao)

    return pd.DataFrame(dados_usuarios)

def load_and_process_data():
    df_misc = pd.read_csv(MISCONCEPTIONS_PATH)
    df_misc['usuario'] = df_misc['usuario'].astype(str)

    for mi in PC3_MAP.values():
        df_misc[f'tem_{mi}'] = False

    # Expande as listas de misconceptions uma única vez e calcula os usuários
    # afetados por grupo com operações vetorizadas do pandas. Isso preserva a
    # regra anterior (a coluna é True quando o usuário possui pelo menos uma
    # ocorrência) sem iterrows nem atribuições linha a linha.
    valid_misconceptions = set(PC3_MAP.values())
    misc_por_usuario = df_misc[['usuario', 'misconceptions_detectados']].copy()
    misc_por_usuario['misconception'] = (
        misc_por_usuario['misconceptions_detectados']
        .fillna('')
        .astype(str)
        .str.split(',')
    )
    misc_por_usuario = misc_por_usuario.explode('misconception')
    misc_por_usuario['misconception'] = misc_por_usuario['misconception'].str.strip()
    misc_por_usuario = misc_por_usuario[
        misc_por_usuario['misconception'].isin(valid_misconceptions)
    ]
    usuarios_por_misconception = (
        misc_por_usuario.groupby('misconception', sort=False)['usuario']
        .unique()
        .to_dict()
    )
    for mi in PC3_MAP.values():
        usuarios_com_mi = usuarios_por_misconception.get(mi, ())
        if len(usuarios_com_mi):
            df_misc.loc[df_misc['usuario'].isin(usuarios_com_mi), f'tem_{mi}'] = True

    df_usuarios = carregar_dados_usuarios()
    return df_misc.merge(df_usuarios, on='usuario', how='left')

def calcular_estatisticas_contingencia(crosstab):
    try:
        chi2, p_val, dof, exp = chi2_contingency(crosstab)
        n = crosstab.sum().sum()
        min_dim = min(crosstab.shape) - 1
        v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
        return chi2, p_val, v
    except:
        return None, None, None

def get_top_misconceptions(df, n=4):
    freqs = {mi: df[f'tem_{mi}'].sum() for mi in PC3_MAP.values() if f'tem_{mi}' in df.columns}
    return [mi for mi, f in sorted(freqs.items(), key=lambda x: x[1], reverse=True)[:n]]

def calcular_cobertura_fator(df, coluna_fator, labels_validos, total_alunos_global, total_codigos_global):
    df_por_usuario = df.drop_duplicates(subset='usuario')
    mask_valido = df_por_usuario[coluna_fator].isin(labels_validos)
    alunos_com = mask_valido.sum()
    alunos_sem = len(df_por_usuario) - alunos_com
    mask_cod = df[coluna_fator].isin(labels_validos)
    codigos_com = mask_cod.sum()
    codigos_sem = len(df) - codigos_com
    pct_alunos  = 100 * alunos_com  / total_alunos_global  if total_alunos_global  > 0 else 0
    pct_codigos = 100 * codigos_com / total_codigos_global if total_codigos_global > 0 else 0
    return {
        'alunos_com':   alunos_com,
        'alunos_sem':   alunos_sem,
        'codigos_com':  codigos_com,
        'codigos_sem':  codigos_sem,
        'pct_alunos':   pct_alunos,
        'pct_codigos':  pct_codigos,
    }

def desenhar_tabela_no_eixo(ax, mi_name, fator_nome, categorias, crosstab, p_value, cramers_v,
                            significativo=True):
    TRADUCAO_CATEGORIAS = {
        'yes': 'Sim', 'no': 'Não', 'public school': 'Pública',
        'private school': 'Privada', 'technical school': 'Técnica'
    }
    categorias_pt  = [TRADUCAO_CATEGORIAS.get(cat, cat) for cat in categorias]
    n_categorias   = len(categorias_pt)
    cor_header   = '#34495e'
    cor_border   = '#2c3e50'
    cor_texto    = '#2c3e50'
    cor_destaque = '#3498db'
    table_top  = 8.0
    cell_width = 1.8 if n_categorias == 2 else 1.5
    cell_height = 1.2
    start_x    = 5 - (cell_width * (n_categorias + 1)) / 2
    largura_total = cell_width * (n_categorias + 1)

    ax.add_patch(FancyBboxPatch((start_x, table_top), largura_total, cell_height,
                                boxstyle="round,pad=0.05", edgecolor=cor_border,
                                facecolor=cor_header, linewidth=2))
    ax.text(start_x + largura_total / 2, table_top + cell_height / 2, fator_nome,
            ha='center', va='center', fontsize=12, fontweight='bold', color='white')

    table_top2 = table_top - cell_height
    ax.add_patch(FancyBboxPatch((start_x, table_top2), cell_width, cell_height,
                                boxstyle="round,pad=0.05", edgecolor=cor_border,
                                facecolor=cor_destaque, linewidth=2))
    ax.text(start_x + cell_width / 2, table_top2 + cell_height / 2, 'PC³',
            ha='center', va='center', fontsize=12, fontweight='bold', color='white')

    for i, cat_nome in enumerate(categorias_pt):
        x_cat = start_x + cell_width + i * cell_width
        ax.add_patch(FancyBboxPatch((x_cat, table_top2), cell_width, cell_height,
                                    boxstyle="round,pad=0.05", edgecolor=cor_border,
                                    facecolor=cor_destaque, linewidth=2))
        ax.text(x_cat + cell_width / 2, table_top2 + cell_height / 2, cat_nome,
                ha='center', va='center', fontsize=11, fontweight='bold', color='white')

    table_top3 = table_top2 - cell_height
    ax.add_patch(FancyBboxPatch((start_x, table_top3), cell_width, cell_height,
                                boxstyle="round,pad=0.05", edgecolor=cor_border,
                                facecolor='#e8f5e9', linewidth=2))
    ax.text(start_x + cell_width / 2, table_top3 + cell_height / 2, f'{mi_name}',
            ha='center', va='center', fontsize=12, fontweight='bold', color=cor_texto)

    for i, cat in enumerate(categorias):
        x_val = start_x + cell_width + i * cell_width
        val   = crosstab.loc[cat, True]  if (cat in crosstab.index and True  in crosstab.columns) else 0
        pct   = (val / crosstab.loc[cat].sum() * 100) if crosstab.loc[cat].sum() > 0 else 0
        ax.add_patch(FancyBboxPatch((x_val, table_top3), cell_width, cell_height,
                                    boxstyle="round,pad=0.05", edgecolor=cor_border,
                                    facecolor='white', linewidth=2))
        ax.text(x_val + cell_width / 2, table_top3 + cell_height / 2,
                f'{formatar_numero(val)}\n({round(pct):.0f}%)',
                ha='center', va='center', fontsize=11, color=cor_texto)

    table_top4 = table_top3 - cell_height
    ax.add_patch(FancyBboxPatch((start_x, table_top4), cell_width, cell_height,
                                boxstyle="round,pad=0.05", edgecolor=cor_border,
                                facecolor='#ffebee', linewidth=2))
    ax.text(start_x + cell_width / 2, table_top4 + cell_height / 2, f'~{mi_name}',
            ha='center', va='center', fontsize=12, fontweight='bold', color=cor_texto)

    for i, cat in enumerate(categorias):
        x_val = start_x + cell_width + i * cell_width
        val   = crosstab.loc[cat, False] if (cat in crosstab.index and False in crosstab.columns) else 0
        pct   = (val / crosstab.loc[cat].sum() * 100) if crosstab.loc[cat].sum() > 0 else 0
        ax.add_patch(FancyBboxPatch((x_val, table_top4), cell_width, cell_height,
                                    boxstyle="round,pad=0.05", edgecolor=cor_border,
                                    facecolor='white', linewidth=2))
        ax.text(x_val + cell_width / 2, table_top4 + cell_height / 2,
                f'{formatar_numero(val)}\n({round(pct):.0f}%)',
                ha='center', va='center', fontsize=11, color=cor_texto)

    stats_width  = 6.0
    stats_height = 1.8
    stats_x = 5 - stats_width / 2
    stats_y = table_top4 - 0.5 - stats_height

    ax.add_patch(FancyBboxPatch((stats_x, stats_y), stats_width, stats_height,
                                boxstyle="round,pad=0.05", edgecolor=cor_border,
                                facecolor='#f8f9fa', linewidth=2))

    if significativo:
        p_str   = formatar_pvalor(p_value)
        v_str   = formatar_numero(cramers_v)
        int_str = ("Muito Forte" if cramers_v > 0.25 else
                   "Forte"       if cramers_v > 0.15 else
                   "Moderado"    if cramers_v > 0.10 else
                   "Fraco"       if cramers_v > 0.05 else "Muito Fraco")
        cor_stat = '#2c3e50'
    else:
        p_str   = formatar_pvalor(p_value) if p_value is not None else 'NS'
        v_str   = 'NS'
        int_str = 'NS'
        cor_stat = '#999999'

    ax.text(5, stats_y + stats_height * 0.75, f'valor-p: {p_str}',
            ha='center', va='center', fontsize=12, fontweight='bold', color=cor_stat)
    ax.text(5, stats_y + stats_height * 0.50, f'V de Cramer: {v_str}',
            ha='center', va='center', fontsize=12, fontweight='bold', color=cor_stat)
    ax.text(5, stats_y + stats_height * 0.25, f'Intensidade: {int_str}',
            ha='center', va='center', fontsize=12, fontweight='bold', color=cor_stat)

    padding = 0.05
    ax.set_xlim(start_x - padding, start_x + largura_total + padding)
    ax.set_ylim(stats_y - padding, table_top + cell_height + padding)
    ax.axis('off')

def desenhar_rodape_cobertura(fig, cob, total_alunos, total_codigos):
    cor_fundo   = '#eaf4fb'
    cor_borda   = '#2980b9'
    cor_titulo  = '#1a5276'
    cor_ok      = '#1e8449'
    cor_falta   = '#922b21'

    ax_rod = fig.add_axes([0.02, 0.01, 0.96, 0.07])
    ax_rod.set_xlim(0, 1)
    ax_rod.set_ylim(0, 1)
    ax_rod.axis('off')

    ax_rod.add_patch(FancyBboxPatch((0, 0), 1, 1,
                                    boxstyle="round,pad=0.01",
                                    edgecolor=cor_borda, facecolor=cor_fundo,
                                    linewidth=1.5, transform=ax_rod.transAxes))

    ax_rod.text(0.5, 0.82, '📊 Cobertura dos dados neste fator',
                ha='center', va='center', fontsize=9, fontweight='bold',
                color=cor_titulo, transform=ax_rod.transAxes)

    txt_alunos_ok   = (f"✔ Alunos com dado: {formatar_numero(int(cob['alunos_com']))} "
                       f"/ {formatar_numero(total_alunos)} "
                       f"({cob['pct_alunos']:.1f}%)")
    txt_alunos_falt = (f"✘ Sem dado: {formatar_numero(int(cob['alunos_sem']))} "
                       f"({100 - cob['pct_alunos']:.1f}%)")
    txt_cod_ok   = (f"✔ Códigos com dado: {formatar_numero(int(cob['codigos_com']))} "
                    f"/ {formatar_numero(total_codigos)} "
                    f"({cob['pct_codigos']:.1f}%)")
    txt_cod_falt = (f"✘ Sem dado: {formatar_numero(int(cob['codigos_sem']))} "
                    f"({100 - cob['pct_codigos']:.1f}%)")

    ax_rod.text(0.02, 0.45, txt_alunos_ok,  ha='left', va='center', fontsize=8.5,
                color=cor_ok,   transform=ax_rod.transAxes, fontweight='bold')
    ax_rod.text(0.30, 0.45, txt_alunos_falt, ha='left', va='center', fontsize=8.5,
                color=cor_falta, transform=ax_rod.transAxes, fontweight='bold')
    ax_rod.text(0.02, 0.15, txt_cod_ok,   ha='left', va='center', fontsize=8.5,
                color=cor_ok,   transform=ax_rod.transAxes, fontweight='bold')
    ax_rod.text(0.30, 0.15, txt_cod_falt,  ha='left', va='center', fontsize=8.5,
                color=cor_falta, transform=ax_rod.transAxes, fontweight='bold')

def gerar_figuras_agrupadas(df, top_misconceptions, grupos_analise,
                             total_alunos_global, total_codigos_global):
    TRADUCAO_FATORES = {
        'school_type':        'Tipo de Escola',
        'previous_experience':'Experiência Prévia com Programação',
        'share_pc':           'Compartilha PC em Casa',
        'worked_interned':    'Trabalhou ou Estagiou Antes do Curso'
    }

    figuras_por_fator = {}

    for grupo in grupos_analise:
        fator_nome       = TRADUCAO_FATORES.get(grupo['column'], grupo['title'])
        resultados_fator = []

        cob = calcular_cobertura_fator(
            df, grupo['column'], grupo['labels'],
            total_alunos_global, total_codigos_global
        )

        for mi_name in top_misconceptions:
            mi_column = f'tem_{mi_name}'
            df_valid  = df[df[grupo['column']].notna()].copy()
            if len(df_valid) > 0 and mi_column in df_valid.columns:
                crosstab = pd.crosstab(df_valid[grupo['column']], df_valid[mi_column])
                if True in crosstab.columns and False in crosstab.columns:
                    labels_presentes = [l for l in grupo['labels'] if l in crosstab.index]
                    if len(labels_presentes) >= 2:
                        crosstab_filtrado = crosstab.loc[labels_presentes]
                        chi2, p_value, cramers_v = calcular_estatisticas_contingencia(crosstab_filtrado)
                        significativo = (p_value is not None and cramers_v is not None
                                         and p_value < 0.05)
                        resultados_fator.append({
                            'mi': mi_name, 'categorias': labels_presentes,
                            'crosstab': crosstab_filtrado,
                            'p_value': p_value, 'cramers_v': cramers_v,
                            'significativo': significativo
                        })

        if resultados_fator:
            resultados_fator = sorted(resultados_fator, key=lambda x: (x['cramers_v'] or 0), reverse=True)

            n_tabelas = len(resultados_fator)
            cols      = min(n_tabelas, 3)
            rows      = (n_tabelas + cols - 1) // cols

            n_cat  = len(grupo['labels'])
            base_w = 6 if n_cat == 2 else 7
            base_h = 7

            fig, axes = plt.subplots(rows, cols,
                                     figsize=(base_w * cols, base_h * rows + 1.0))
            fig.subplots_adjust(bottom=0.10)

            if n_tabelas == 1:
                axes = np.array([axes])
            axes_flat = axes.flatten()

            for i, res in enumerate(resultados_fator):
                desenhar_tabela_no_eixo(
                    axes_flat[i], res['mi'], fator_nome, res['categorias'],
                    res['crosstab'], res['p_value'], res['cramers_v'],
                    significativo=res['significativo']
                )

            for j in range(i + 1, len(axes_flat)):
                axes_flat[j].axis('off')

            desenhar_rodape_cobertura(fig, cob, total_alunos_global, total_codigos_global)

            plt.tight_layout(rect=[0, 0.09, 1, 1])
            figuras_por_fator[fator_nome] = fig

            print(f"\n  [{fator_nome}]")
            print(f"    Alunos  com dado: {cob['alunos_com']:>6} / {total_alunos_global} "
                  f"({cob['pct_alunos']:.1f}%)  |  sem dado: {cob['alunos_sem']} "
                  f"({100-cob['pct_alunos']:.1f}%)")
            print(f"    Códigos com dado: {cob['codigos_com']:>6} / {total_codigos_global} "
                  f"({cob['pct_codigos']:.1f}%)  |  sem dado: {cob['codigos_sem']} "
                  f"({100-cob['pct_codigos']:.1f}%)")

    return figuras_por_fator


# =============================================================================
# GERAÇÃO DE CÓDIGO LATEX ORDENADO PELO MENOR P-VALOR
# =============================================================================

def formatar_pvalor_latex(p_val):
    if p_val is None:
        return 'NS'
    if p_val < 1e-16:
        return r'$\ll$ 0,001'
    elif p_val < 1e-3:
        return r'$<$ 0,001'
    else:
        return f"{p_val:.3f}".replace('.', ',')

def formatar_cramers_latex(v, significativo):
    if not significativo or v is None:
        return 'NS'
    return f"{v:.3f}".replace('.', ',')

def intensidade_latex(v, significativo):
    if not significativo or v is None:
        return 'NS'
    if v > 0.25: return 'Muito forte'
    if v > 0.15: return 'Forte'
    if v > 0.10: return 'Moderado'
    if v > 0.05: return 'Fraco'
    return 'Muito fraco'

def calcular_dados_latex(df, grupo, top_misconceptions):
    resultados = []
    for mi_name in top_misconceptions:
        mi_column = f'tem_{mi_name}'
        df_valid  = df[df[grupo['column']].notna()].copy()
        if len(df_valid) == 0 or mi_column not in df_valid.columns:
            continue

        crosstab = pd.crosstab(df_valid[grupo['column']], df_valid[mi_column])
        if True not in crosstab.columns or False not in crosstab.columns:
            continue

        labels_presentes = [l for l in grupo['labels'] if l in crosstab.index]
        if len(labels_presentes) < 2:
            continue

        crosstab_f = crosstab.loc[labels_presentes]
        chi2, p_value, cramers_v = calcular_estatisticas_contingencia(crosstab_f)
        significativo = (p_value is not None and cramers_v is not None and p_value < 0.05)

        cats = {}
        for label in labels_presentes:
            total_cat = crosstab_f.loc[label].sum()
            val_pres  = int(crosstab_f.loc[label, True])  if True  in crosstab_f.columns else 0
            val_aus   = int(crosstab_f.loc[label, False]) if False in crosstab_f.columns else 0
            pct_pres  = round(100 * val_pres / total_cat) if total_cat > 0 else 0
            pct_aus   = 100 - pct_pres
            cats[label] = {
                'presente': val_pres, 'pct_pres': pct_pres,
                'ausente':  val_aus,  'pct_aus':  pct_aus,
            }

        resultados.append({
            'mi':            mi_name,
            'labels':        labels_presentes,
            'cats':          cats,
            'p_value':       p_value if p_value is not None else 1.0, # Atribui 1.0 para jogar NS para o fim
            'cramers_v':     cramers_v,
            'significativo': significativo,
        })

    # CORREÇÃO DA ORDENAÇÃO: Ordena explicitamente pelo menor p-valor
    resultados.sort(key=lambda x: x['p_value'])
    return resultados


def gerar_tablenotes(alunos_sem, codigos_sem, tem_ns=True):
    itens = []
    if alunos_sem > 0 or codigos_sem > 0:
        n_alunos  = formatar_numero(int(alunos_sem))
        n_codigos = formatar_numero(int(codigos_sem))
        itens.append(
            f"        \\item Informações de {n_alunos} estudante(s) e "
            f"{n_codigos} código(s) submetido(s) não puderam ser "
            f"recuperadas no repositório no momento da coleta e não "
            f"foram incluídas nesta análise."
        )
    itens.append(r"        \item NS: não significativo")
    corpo = "\n".join(itens)
    return f"\n    \\begin{{tablenotes}}\n{corpo}\n    \\end{{tablenotes}}"


def gerar_latex_escola(df, top_misconceptions, alunos_sem=0, codigos_sem=0):
    grupo = {
        'column': 'school_type',
        'labels': ('public school', 'private school', 'technical school'),
    }
    dados = calcular_dados_latex(df, grupo, top_misconceptions)
    tem_ns = any(not d['significativo'] for d in dados)

    linhas = []
    for i, d in enumerate(dados):
        PCT = r'\%'
        if i > 0:
            linhas.append(r'        \midrule')
        cats = d['cats']
        pub  = cats.get('public school',    {'presente': 0, 'pct_pres': 0, 'ausente': 0, 'pct_aus': 0})
        pri  = cats.get('private school',   {'presente': 0, 'pct_pres': 0, 'ausente': 0, 'pct_aus': 0})
        tec  = cats.get('technical school', {'presente': 0, 'pct_pres': 0, 'ausente': 0, 'pct_aus': 0})

        # Restaura o p-valor real para exibição se ele foi mascarado para ordenação
        p_real = None if d['p_value'] == 1.0 and not d['significativo'] else d['p_value']
        p_tex = formatar_pvalor_latex(p_real)
        v_tex = formatar_cramers_latex(d['cramers_v'], d['significativo'])
        i_tex = intensidade_latex(d['cramers_v'], d['significativo'])

        linhas.append(
            f"        {d['mi']} & Presente & "
            f"{formatar_numero(pub['presente'])} ({pub['pct_pres']}{PCT}) & "
            f"{formatar_numero(pri['presente'])} ({pri['pct_pres']}{PCT}) & "
            f"{formatar_numero(tec['presente'])} ({tec['pct_pres']}{PCT}) & "
            f"{p_tex} & {v_tex} & {i_tex} \\\\"
        )
        linhas.append(
            f"           & Ausente & "
            f"{formatar_numero(pub['ausente'])} ({pub['pct_aus']}{PCT}) & "
            f"{formatar_numero(pri['ausente'])} ({pri['pct_aus']}{PCT}) & "
            f"{formatar_numero(tec['ausente'])} ({tec['pct_aus']}{PCT}) & & & \\\\"
        )

    corpo      = "\n".join(linhas)
    tablenotes = gerar_tablenotes(alunos_sem, codigos_sem, tem_ns=tem_ns)

    return (
        r"\begin{table}[H]" "\n"
        r"    \centering\footnotesize" "\n"
        r"    \setlength{\tabcolsep}{6pt}" "\n"
        r"    \begin{threeparttable}" "\n"
        r"    \caption{Quantidade de códigos com presença/ausência de PC$^3$, "
        r"distribuídos conforme o tipo de escola do Ensino Médio de origem do aluno "
        r"(convencional pública, convencional privada ou técnica). Fonte: }" "\n"
        r"    \label{tab:misconceptions_escola}" "\n"
        r"    \begin{tabular}{ll rrr ccc}" "\n"
        r"        \toprule" "\n"
        r"        \textbf{PC$^3$} & \textbf{\vtop{\hbox{Presença}\hbox{no código}}} & \textbf{Pública} & "
        r"\textbf{Privada} & \textbf{Técnica} & \textbf{$p$-valor} & "
        r"\textbf{V de Cramer} & \textbf{Intensidade} \\" "\n"
        r"        \midrule" "\n"
        f"{corpo}" "\n"
        r"        \bottomrule" "\n"
        r"    \end{tabular}" "\n"
        f"{tablenotes}" "\n"
        r"    \end{threeparttable}" "\n"
        r"\end{table}"
    )


def gerar_latex_dois_grupos(df, top_misconceptions, grupo, caption, label, col_sim, col_nao,
                            alunos_sem=0, codigos_sem=0):
    dados = calcular_dados_latex(df, grupo, top_misconceptions)
    if not dados:
        return None

    tem_ns = any(not d['significativo'] for d in dados)
    label_sim = grupo['labels'][0]
    label_nao = grupo['labels'][1]

    linhas = []
    for i, d in enumerate(dados):
        PCT = r'\%'
        if i > 0:
            linhas.append(r'        \midrule')
        cats = d['cats']
        sim  = cats.get(label_sim, {'presente': 0, 'pct_pres': 0, 'ausente': 0, 'pct_aus': 0})
        nao  = cats.get(label_nao, {'presente': 0, 'pct_pres': 0, 'ausente': 0, 'pct_aus': 0})

        p_real = None if d['p_value'] == 1.0 and not d['significativo'] else d['p_value']
        p_tex = formatar_pvalor_latex(p_real)
        v_tex = formatar_cramers_latex(d['cramers_v'], d['significativo'])
        i_tex = intensidade_latex(d['cramers_v'], d['significativo'])

        linhas.append(
            f"        {d['mi']} & Presente & "
            f"{formatar_numero(sim['presente'])} ({sim['pct_pres']}{PCT}) & "
            f"{formatar_numero(nao['presente'])} ({nao['pct_pres']}{PCT}) & "
            f"{p_tex} & {v_tex} & {i_tex} \\\\"
        )
        linhas.append(
            f"           & Ausente & "
            f"{formatar_numero(sim['ausente'])} ({sim['pct_aus']}{PCT}) & "
            f"{formatar_numero(nao['ausente'])} ({nao['pct_aus']}{PCT}) & & & \\\\"
        )

    corpo      = "\n".join(linhas)
    tablenotes = gerar_tablenotes(alunos_sem, codigos_sem, tem_ns=tem_ns)

    return (
        r"\begin{table}[H]" "\n"
        r"    \centering\footnotesize" "\n"
        r"    \setlength{\tabcolsep}{7pt}" "\n"
        r"    \begin{threeparttable}" "\n"
        f"    \\caption{{{caption}}}" "\n"
        f"    \\label{{{label}}}" "\n"
        r"    \begin{tabular}{ll rr ccc}" "\n"
        r"        \toprule" "\n"
        f"        \\textbf{{PC$^3$}} & \\textbf{{\\vtop{{\\hbox{{Presença}}\\hbox{{no código}}}}}} & \\textbf{{{col_sim}}} & "
        f"\\textbf{{{col_nao}}} & \\textbf{{$p$-valor}} & "
        r"\textbf{V de Cramer} & \textbf{Intensidade} \\" "\n"
        r"        \midrule" "\n"
        f"        {corpo}" "\n"
        r"        \bottomrule" "\n"
        r"    \end{tabular}" "\n"
        f"{tablenotes}" "\n"
        r"    \end{threeparttable}" "\n"
        r"\end{table}"
    )


def gerar_todos_latex(df, top_misconceptions, arquivo_saida="tabelas_latex.tex"):
    blocos = []
    total_alunos  = df['usuario'].nunique()
    total_codigos = len(df)

    def _cob(column, labels):
        df_u  = df.drop_duplicates(subset='usuario')
        a_com = df_u[column].isin(labels).sum()
        c_com = df[column].isin(labels).sum()
        return int(total_alunos - a_com), int(total_codigos - c_com)

    # Tabela 1: Tipo de escola
    labels_escola = ('public school', 'private school', 'technical school')
    a_sem_esc, c_sem_esc = _cob('school_type', labels_escola)
    blocos.append(gerar_latex_escola(df, top_misconceptions,
                                     alunos_sem=a_sem_esc, codigos_sem=c_sem_esc))

    # Tabela 2: Compartilha PC em casa
    grupo_pc = {'column': 'share_pc', 'labels': ('yes', 'no')}
    a_sem_pc, c_sem_pc = _cob('share_pc', ('yes', 'no'))
    tex_pc = gerar_latex_dois_grupos(
        df, top_misconceptions, grupo_pc,
        caption="Distribuição dos PC$^3$ por compartilhamento de computador pessoal em casa",
        label="tab:misconceptions_pc_casa",
        col_sim="Sim", col_nao="Não",
        alunos_sem=a_sem_pc, codigos_sem=c_sem_pc
    )
    if tex_pc: blocos.append(tex_pc)

    # Tabela 3: Experiencia previa em programacao
    grupo_exp = {'column': 'previous_experience', 'labels': ('yes', 'no')}
    a_sem_exp, c_sem_exp = _cob('previous_experience', ('yes', 'no'))
    tex_exp = gerar_latex_dois_grupos(
        df, top_misconceptions, grupo_exp,
        caption="Distribuição dos PC$^3$ por experiência prévia em programação",
        label="tab:misconceptionsExperienciaPrevia",
        col_sim="Sim", col_nao="Não",
        alunos_sem=a_sem_exp, codigos_sem=c_sem_exp
    )
    if tex_exp: blocos.append(tex_exp)

    # Tabela 4: Trabalhou/estagiou antes do curso
    grupo_trab = {'column': 'worked_interned', 'labels': ('yes', 'no')}
    a_sem_trab, c_sem_trab = _cob('worked_interned', ('yes', 'no'))
    tex_trab = gerar_latex_dois_grupos(
        df, top_misconceptions, grupo_trab,
        caption="Distribuição dos PC$^3$ por experiência profissional prévia",
        label="tab:misconceptions_experiencia_profissional",
        col_sim="Sim", col_nao="Não",
        alunos_sem=a_sem_trab, codigos_sem=c_sem_trab
    )
    if tex_trab: blocos.append(tex_trab)

    conteudo = "\n\n".join(str(b) for b in blocos)
    with open(arquivo_saida, 'w', encoding='utf-8') as f:
        f.write(conteudo)

    print(f"\n Código LaTeX salvo em: {arquivo_saida}")
    return conteudo


if __name__ == "__main__":
    print("\n" + "=" * 80)
    print("ANÁLISE DE CONTINGÊNCIA - FIGURAS AGRUPADAS")
    print("=" * 80 + "\n")

    df_final = load_and_process_data()
    top_misconceptions = get_top_misconceptions(df_final, n=4)

    total_alunos_global  = df_final['usuario'].nunique()
    total_codigos_global = len(df_final)

    print(f"Total de registros carregados : {formatar_numero(total_codigos_global)}")
    print(f"Usuários únicos               : {formatar_numero(total_alunos_global)}")
    print(f"\nPC3 analisados ({len(top_misconceptions)}):")
    for mi in top_misconceptions:
        freq = df_final[f'tem_{mi}'].sum()
        print(f"   {mi}: {formatar_numero(int(freq))} ocorrências")

    grupos_analise = [
        {'column': 'school_type',
         'labels': ('public school', 'private school', 'technical school'),
         'title':  'Tipo de Escola'},
        {'column': 'previous_experience',
         'labels': ('yes', 'no'),
         'title':  'Experiencia Previa'},
        {'column': 'share_pc',
         'labels': ('yes', 'no'),
         'title':  'Compartilha PC'},
        {'column': 'worked_interned',
         'labels': ('yes', 'no'),
         'title':  'Trabalhou/Estagiou'},
    ]

    print("\nGerando fingerprints / figuras de agrupamento...")
    figuras_agrupadas = gerar_figuras_agrupadas(
        df_final, top_misconceptions, grupos_analise,
        total_alunos_global, total_codigos_global
    )

    if figuras_agrupadas:
        print(f"\n{'='*80}")
        print(f"TOTAL DE FIGURAS GERADAS: {len(figuras_agrupadas)}")
        print(f"{'='*80}\n")

        for fator, fig in figuras_agrupadas.items():
            nome_arquivo = f"contingencia_{fator.replace(' ', '_').lower()}.png"
            fig.savefig(nome_arquivo, bbox_inches='tight', pad_inches=0.1, dpi=300)
            print(f"Salvo: {nome_arquivo}")
            plt.show()
    else:
        print("\nNenhuma figura foi gerada. Verifique se os dados dos usuários estão disponíveis.")

    print("\nGerando tabelas LaTeX estruturadas...")
    gerar_todos_latex(df_final, top_misconceptions, arquivo_saida="tabelas_latex.tex")

    print("\n" + "=" * 80)
    print("ANÁLISE FINALIZADA COM SUCESSO")
    print("=" * 80)

# Etapa 7

In [ ]:
"""
COMBINADOR DE MÉTRICAS - BASEADO EM MISCONCEPTIONS
Usa output/misconceptions_detalhado_por_usuario.csv como fonte principal

MODIFICAÇÕES:
1. nota_aluno é calculada mas NÃO incluída na saída
2. dificuldade é calculada da MESMA forma (do arquivo .log individual)
3. OTIMIZADO: Acesso a disco reduzido, Regex pré-compilado e GroupBy utilizado.
4. NOVO: adicionadas as versões individuais (por episódio) de 3 métricas que
   em output/dataset_analise_questoes.csv só existiam agregadas por questão:
   - taxa_acerto     (M1): = num_correct do próprio episódio (denominador
     "num_students_interactions" do aluno é sempre 1, então degenera em
     num_correct — mantida como coluna própria por nomenclatura/compatibilidade).
   - taxa_aceitacao  (M3): = num_correct / num_submissoes DO PRÓPRIO ALUNO
     (fração das tentativas dele que deram certo — ex.: acertar de primeira
     vs. acertar só na 38ª submissão).
   - num_consultas   (M5): = num_submissoes + num_tests do próprio episódio
     (ANTES ficava sempre NaN, nunca era calculada).
   A métrica M13 (discriminacao) continua fora: é uma estatística sobre o
   conjunto de alunos de uma questão e não tem equivalente por episódio.
"""

import pandas as pd
import numpy as np
import os
import re
import glob

# ============================================================================
# CONFIGURAÇÕES
# ============================================================================

MISCONCEPTIONS_PATH = 'output/misconceptions_detalhado_por_usuario.csv'
EXECUTIONS_PATH = '../Etapa_4/codebench-analytics-full/output/metrics/executions_by_student.csv'
ACTIONS_PATH = '../Etapa_4/codebench-analytics-full/output/metrics/actions_by_student.csv'
USUARIOS_PATH = '../Etapa_2/output/referencias_processamento.json'

OUTPUT_PATH = 'output/metricas_execucao_por_estudante_questao.csv'
os.makedirs('output', exist_ok=True)

# OTIMIZAÇÃO: Pré-compilar o Regex UMA ÚNICA VEZ fora da função
PADRAO_NOTA = re.compile(r'--\s*GRADE:\s*(\d+)\s*%', re.MULTILINE)

# ============================================================================
# FUNÇÃO: EXTRAIR DIFICULDADE (OTIMIZADA)
# ============================================================================

def obter_nota_arquivo(arquivo_path: str) -> int:
    """Extrai a nota do arquivo .log (retorna percentual 0-100)"""
    try:
        with open(arquivo_path, 'r', encoding='utf-8') as file:
            conteudo = file.read()
            # Usar o Regex pré-compilado (muito mais rápido)
            notas = PADRAO_NOTA.findall(conteudo)
            if notas:
                # Retornar a ÚLTIMA nota encontrada (submissão final)
                return int(notas[-1])
    except Exception:
        pass
    return 0


def extrair_dificuldades_para_alunos_questoes(caminho_usuarios, df_misconceptions):
    """
    Extrai dificuldade (nota do .log) APENAS para os pares (aluno, questão) que existem em misconceptions
    Versão Otimizada: Usa operações em lote e busca em memória para reduzir uso do disco (I/O).
    """
    
    print("\n📝 Extraindo dificuldades para pares (aluno, questão) em misconceptions...")
    
    if not os.path.exists(caminho_usuarios):
        print(f"❌ JSON de referências não encontrado: {caminho_usuarios}")
        return None, None
    manifest = carregar_referencias(caminho_usuarios)
    
    # Extrair pares únicos (aluno, questão) de misconceptions
    pares_aluno_questao = df_misconceptions[['usuario', 'question']].drop_duplicates()
    print(f"   • Pares únicos (aluno, questão): {len(pares_aluno_questao)}")
    
    dificuldades_individuais = []
    arquivos_lidos = {}  # {(usuario, questao): caminho}
    
    arquivos_encontrados = 0
    arquivos_nao_encontrados = 0
    dificuldades_extraidas = 0
    
    # OTIMIZAÇÃO: Agrupar por aluno para varrer o diretório apenas uma vez por pessoa
    questoes_por_aluno = pares_aluno_questao.groupby('usuario')['question'].apply(list).to_dict()

    for aluno, questoes in questoes_por_aluno.items():
        aluno_str = str(aluno)
        caminhos_aluno = list(iterar_arquivos_usuario(manifest, aluno_str, 'executions', '.log'))
        if not caminhos_aluno:
            caminhos_aluno = list(iterar_arquivos_usuario(manifest, aluno_str, 'grades', '.log'))
        if not caminhos_aluno:
            arquivos_nao_encontrados += len(questoes)
            continue
        arquivos_por_nome = {p.name: p for p in caminhos_aluno}

        # Iterar apenas nas questões deste aluno
        for questao in questoes:
            questao_str = str(questao)
            caminho_arquivo_log = None
            
            # OTIMIZAÇÃO: Busca na memória (set intersection) é instantânea
            possiveis_exatos = {f'{questao_str}.log', f'execution_{questao_str}.log', f'{aluno_str}_{questao_str}.log'}
            match_exato = set(arquivos_por_nome).intersection(possiveis_exatos)
            
            if match_exato:
                caminho_arquivo_log = str(arquivos_por_nome[next(iter(match_exato))])
            else:
                # Busca pelo sufixo *_QUESTAO.log na memória
                sufixo_1 = f'_{questao_str}.log'
                for arq, path in arquivos_por_nome.items():
                    if arq.endswith(sufixo_1):
                        caminho_arquivo_log = str(path)
                        break
            
            if caminho_arquivo_log:
                nota_percentual = obter_nota_arquivo(caminho_arquivo_log)
                arquivos_lidos[(aluno_str, questao)] = caminho_arquivo_log
                
                dificuldades_individuais.append({
                    'usuario': aluno_str,
                    'question': questao,
                    'dificuldade': nota_percentual
                })
                
                arquivos_encontrados += 1
                if nota_percentual > 0:
                    dificuldades_extraidas += 1
            else:
                arquivos_nao_encontrados += 1
    
    print(f"\n✅ Resultado da extração:")
    print(f"   • Arquivos .log encontrados: {arquivos_encontrados}")
    print(f"   • Arquivos .log NÃO encontrados: {arquivos_nao_encontrados}")
    print(f"   • Dificuldades extraídas (>0): {dificuldades_extraidas}")
    print(f"   • Total de registros processados: {len(dificuldades_individuais)}")
    
    if (arquivos_encontrados + arquivos_nao_encontrados) > 0:
        taxa = arquivos_encontrados / (arquivos_encontrados + arquivos_nao_encontrados) * 100
        print(f"   • Taxa de sucesso: {taxa:.1f}%")
    
    df_dificuldades_individuais = pd.DataFrame(dificuldades_individuais)
    
    return df_dificuldades_individuais, arquivos_lidos


def normalizar_id_usuario(df, coluna='usuario'):
    """Normaliza IDs de usuário"""
    df = df.copy()
    df[coluna] = df[coluna].astype(str)
    df[coluna] = df[coluna].str.replace(r'^(user_|student_|aluno_)', '', regex=True)
    df[coluna] = df[coluna].str.strip().str.lower()
    return df


# ============================================================================
# FUNÇÃO PRINCIPAL
# ============================================================================

def combinar_metricas():
    """Combina métricas baseado em misconceptions"""
    
    print("\n" + "="*80)
    print("COMBINADOR DE MÉTRICAS - BASEADO EM MISCONCEPTIONS")
    print("="*80)
    print("\n⚠️  MODIFICAÇÕES:")
    print("   1. nota_aluno NÃO será incluída na saída")
    print("   2. dificuldade = valor do arquivo .log (individual)")
    print("="*80)
    
    # ========================================================================
    # PASSO 1: Carregar MISCONCEPTIONS (fonte principal)
    # ========================================================================
    
    print("\n📂 Carregando misconceptions (fonte principal)...")
    
    try:
        df_misconceptions = pd.read_csv(MISCONCEPTIONS_PATH)
        print(f"✅ Misconceptions: {len(df_misconceptions)} registros")
        
        # Normalizar IDs
        df_misconceptions = normalizar_id_usuario(df_misconceptions, 'usuario')
        
        # Extrair listas de alunos e questões elegíveis
        alunos_elegiveis = sorted(df_misconceptions['usuario'].unique())
        questoes_elegiveis = sorted(df_misconceptions['question'].unique())
        
        print(f"\n📊 ALUNOS E QUESTÕES ELEGÍVEIS:")
        print(f"   • Total de alunos únicos: {len(alunos_elegiveis)}")
        print(f"   • Total de questões únicas: {len(questoes_elegiveis)}")
        
        print(f"\n📝 Primeiros 20 alunos elegíveis:")
        print(f"   {', '.join(map(str, alunos_elegiveis[:20]))}")
        
        print(f"\n📝 Primeiras 20 questões elegíveis:")
        print(f"   {', '.join(map(str, questoes_elegiveis[:20]))}")
        
    except Exception as e:
        print(f"❌ Erro ao carregar misconceptions: {e}")
        return None
    
    # ========================================================================
    # PASSO 2: FILTRAR Executions pelos pares (student, question) de misconceptions
    # ========================================================================
    
    print("\n📂 PASSO 2: Filtrando Executions...")
    
    try:
        df_executions = pd.read_csv(EXECUTIONS_PATH)
        df_executions = df_executions.rename(columns={'student': 'usuario'})
        df_executions = normalizar_id_usuario(df_executions)
        print(f"✅ Executions lido: {len(df_executions)} registros")
        
        # Criar lista de pares válidos (usuario, question) de misconceptions
        pares_validos = df_misconceptions[['usuario', 'question']].drop_duplicates()
        print(f"   • Pares válidos em misconceptions: {len(pares_validos)}")
        
        # FILTRAR: manter APENAS registros cujo par (usuario, question) existe em misconceptions
        df_executions_filtrado = pares_validos.merge(
            df_executions,
            on=['usuario', 'question'],
            how='inner'
        )
        
        registros_removidos = len(df_executions) - len(df_executions_filtrado)
        print(f"   • Registros MANTIDOS: {len(df_executions_filtrado)}")
        print(f"   • Registros REMOVIDOS: {registros_removidos}")
        
        df_executions = df_executions_filtrado
        
    except Exception as e:
        print(f"❌ Erro ao carregar executions: {e}")
        return None
    
    # ========================================================================
    # PASSO 3: FILTRAR Actions pelos pares (student, question) de misconceptions
    # ========================================================================
    
    print("\n📂 PASSO 3: Filtrando Actions...")
    
    try:
        df_actions = pd.read_csv(ACTIONS_PATH)
        df_actions = df_actions.rename(columns={'student': 'usuario'})
        df_actions = normalizar_id_usuario(df_actions)
        print(f"✅ Actions lido: {len(df_actions)} registros")
        
        # FILTRAR: manter APENAS registros cujo par (usuario, question) existe em misconceptions
        df_actions_filtrado = pares_validos.merge(
            df_actions,
            on=['usuario', 'question'],
            how='inner'
        )
        
        registros_removidos = len(df_actions) - len(df_actions_filtrado)
        print(f"   • Registros MANTIDOS: {len(df_actions_filtrado)}")
        print(f"   • Registros REMOVIDOS: {registros_removidos}")
        
        df_actions = df_actions_filtrado
        
    except Exception as e:
        print(f"❌ Erro ao carregar actions: {e}")
        return None
    
    # ========================================================================
    # PASSO 4: Extrair dificuldades para os pares (aluno, questão) válidos
    # ========================================================================
    
    print("\n📂 PASSO 4: Extraindo dificuldades (do .log)...")
    
    df_dificuldades, arquivos_lidos = extrair_dificuldades_para_alunos_questoes(
        USUARIOS_PATH, 
        df_misconceptions
    )
    
    if df_dificuldades is None:
        print("\n❌ ERRO: Não foi possível extrair dificuldades!")
        return None
    
    # ========================================================================
    # PASSO 5: Combinar tudo - Usar misconceptions como BASE
    # ========================================================================
    
    print("\n🔗 PASSO 5: Combinando métricas...")
    
    # Começar com os pares de misconceptions
    df_combined = df_misconceptions.copy()
    print(f"   • Base (misconceptions): {len(df_combined)} registros")
    
    # Adicionar executions (left join)
    df_combined = df_combined.merge(
        df_executions,
        on=['usuario', 'question'],
        how='left'
    )
    print(f"   • Após adicionar executions: {len(df_combined)} registros")
    
    # Adicionar actions (left join)
    df_combined = df_combined.merge(
        df_actions,
        on=['usuario', 'question'],
        how='left',
        suffixes=('_exec', '_action')
    )
    print(f"   • Após adicionar actions: {len(df_combined)} registros")
    
    # Resolver is_correct duplicado
    if 'is_correct_exec' in df_combined.columns and 'is_correct_action' in df_combined.columns:
        df_combined['is_correct'] = df_combined['is_correct_exec'].fillna(df_combined['is_correct_action'])
        df_combined = df_combined.drop(columns=['is_correct_exec', 'is_correct_action'])
    
    # ========================================================================
    # PASSO 6: Adicionar dificuldades individuais
    # ========================================================================
    
    print("\n🔗 PASSO 6: Adicionando dificuldades individuais...")
    
    # Adicionar dificuldades
    df_combined = df_combined.merge(
        df_dificuldades,
        on=['usuario', 'question'],
        how='left'
    )
    
    dificuldades_validas = df_combined['dificuldade'].notna().sum()
    pct = (dificuldades_validas / len(df_combined)) * 100 if len(df_combined) > 0 else 0
    print(f"✅ Dificuldades individuais: {dificuldades_validas} ({pct:.1f}%)")
    
    # ========================================================================
    # PASSO 7: Criar DataFrame Final (SEM nota_aluno)
    # ========================================================================
    
    print("\n📊 Criando DataFrame final...")
    print(f"   • Total de registros: {len(df_combined)} (mesmo que misconceptions!)")
    
    df_final = pd.DataFrame()
    
    # Colunas de identificação
    df_final['usuario'] = df_combined['usuario'].astype(str)
    df_final['question'] = df_combined['question'].astype(int)
    
    # Métricas brutas do episódio (contagens individuais, sem dividir por nada)
    df_final['tempo_implementacao'] = df_combined['code_time']
    df_final['num_eventos'] = df_combined['num_events']
    df_final['num_eventos_del'] = df_combined['num_deletes']
    df_final['num_submissoes'] = df_combined['num_submissions']
    df_final['num_tests'] = df_combined['num_tests']
    df_final['num_correct'] = df_combined['is_correct'].astype(float)
    df_final['num_errors'] = df_combined['num_errors']
    df_final['num_logic_errors'] = df_combined['num_logic_errors']
    df_final['num_syntax_errors'] = df_combined['num_syntax_errors']
    df_final['qtd_alteracoes_codigo'] = df_combined['amount_of_change']
    
    # ========================================================================
    # MÉTRICAS DERIVADAS NO NÍVEL DE EPISÓDIO — equivalentes individuais de
    # M1, M3 e M5 de output/dataset_analise_questoes.csv (lá, calculadas por QUESTÃO dividindo
    # pelo total de interações/submissões da turma inteira). Aqui, o
    # denominador é sempre o do PRÓPRIO aluno naquele episódio.
    # M13 (discriminacao) não entra: é uma estatística sobre a turma toda e
    # não tem equivalente individual.
    # ========================================================================
    
    # M1 individual — taxa_acerto: no nível de questão é
    # num_correct / num_students_interactions (proporção da TURMA que acertou).
    # No episódio, o equivalente correto NÃO é num_correct (que é binário:
    # 0/1, só diz se passou em TODOS os testes de uma vez). O que representa
    # "quanto da questão o aluno acertou" já existe no pipeline: é a nota em
    # percentual (0-100) extraída do .log via obter_nota_arquivo()/GRADE: XX%,
    # mesclada em df_combined['dificuldade'] no PASSO 6. taxa_acerto é essa
    # nota convertida para a mesma escala 0-1 usada em output/dataset_analise_questoes.csv.
    # (ERRO ANTERIOR: esta linha usava df_final['num_correct'], que é binário
    # e não captura acerto parcial — corrigido para usar a nota percentual.)
    df_final['taxa_acerto'] = df_combined['dificuldade'] / 100
    
    # M3 individual — taxa_aceitacao: no nível de questão é
    # num_correct / num_submissions da turma inteira. No episódio, vira a
    # fração das PRÓPRIAS submissões do aluno que terminaram corretas —
    # esta, sim, é informação nova (ex.: acertar de primeira vs. só na 38ª
    # tentativa). NaN quando o aluno não fez nenhuma submissão.
    df_final['taxa_aceitacao'] = np.where(
        df_final['num_submissoes'] > 0,
        df_final['num_correct'] / df_final['num_submissoes'],
        np.nan
    )
    
    # M5 individual — num_consultas: no nível de questão é
    # (num_submissions + num_tests) / num_students_interactions da turma.
    # No episódio, o denominador é 1, então vira a soma bruta das consultas
    # do próprio aluno. ANTES: ficava sempre NaN (nunca era calculada).
    df_final['num_consultas'] = df_final['num_submissoes'] + df_final['num_tests']
    
    # Misconceptions
    df_final['misconceptions_detectados'] = df_combined['misconceptions_detectados']
    df_final['total_misconceptions'] = df_combined['total_misconceptions']
    df_final['categorias_afetadas'] = df_combined['categorias_afetadas']
    
    # ⚠️ MUDANÇA: Apenas dificuldade (sem nota_aluno)
    df_final['dificuldade'] = df_combined['dificuldade']
    
    print(f"✅ DataFrame final criado: {len(df_final.columns)} colunas")
    print(f"   ✅ Métricas individuais adicionadas: taxa_acerto, taxa_aceitacao, num_consultas")
    print(f"   ℹ️  Agora são 12 das 13 métricas de dataset_analise_questoes.csv disponíveis por episódio")
    print(f"      (falta apenas 'discriminacao', que não tem equivalente individual)")
    print(f"   ⚠️  nota_aluno NÃO foi incluída!")
    
    # ========================================================================
    # PASSO 8: Validação
    # ========================================================================
    
    print("\n" + "="*80)
    print("VALIDAÇÃO")
    print("="*80)
    
    print(f"\n📋 Colunas do arquivo final ({len(df_final.columns)}):")
    for i, col in enumerate(df_final.columns):
        print(f"   {i+1:2d}. {col}")
    
    print(f"\n📊 Estatísticas:")
    print(f"   • Total de registros: {len(df_final)}")
    print(f"   • Alunos únicos: {df_final['usuario'].nunique()}")
    print(f"   • Questões únicas: {df_final['question'].nunique()}")
    
    print(f"\n📊 Dificuldade:")
    print(f"   • Válidos: {df_final['dificuldade'].notna().sum()}")
    print(f"   • NaN: {df_final['dificuldade'].isna().sum()}")
    if df_final['dificuldade'].notna().sum() > 0:
        print(f"   • Média: {df_final['dificuldade'].mean():.2f}")
        print(f"   • Mínimo: {df_final['dificuldade'].min():.2f}")
        print(f"   • Máximo: {df_final['dificuldade'].max():.2f}")
    
    # ========================================================================
    # PASSO 9: Relatório de NaN
    # ========================================================================
    
    print("\n" + "="*80)
    print("ANÁLISE DE DIFICULDADES FALTANTES")
    print("="*80)
    
    total_registros = len(df_final)
    registros_com_dificuldade = df_final['dificuldade'].notna().sum()
    registros_sem_dificuldade = df_final['dificuldade'].isna().sum()
    
    print(f"\n📊 RESUMO GERAL:")
    print(f"   • Total de registros válidos (de misconceptions): {total_registros}")
    print(f"   • Registros COM dificuldade encontrada: {registros_com_dificuldade}")
    print(f"   • Registros SEM dificuldade: {registros_sem_dificuldade}")
    print(f"   • Porcentagem de SUCESSO: {(registros_com_dificuldade/total_registros*100):.1f}%")
    print(f"   • Porcentagem de FALHA: {(registros_sem_dificuldade/total_registros*100):.1f}%")
    
    if registros_sem_dificuldade > 0:
        print(f"\n⚠️  ATENÇÃO: {registros_sem_dificuldade} registros sem dificuldade!")
        print(f"   Estes registros podem ter misconceptions detectados mas sem arquivo .log")
    else:
        print("\n✅ PERFEITO! Todos os registros têm dificuldade!")
    
    # ========================================================================
    # PASSO 10: Amostra e Salvar
    # ========================================================================
    
    print("\n" + "="*80)
    print("AMOSTRA DOS DADOS (15 primeiras linhas)")
    print("="*80)
    
    colunas_amostra = ['usuario', 'question', 'num_correct', 'taxa_acerto', 'taxa_aceitacao', 'num_consultas', 'dificuldade', 'total_misconceptions']
    print(df_final[colunas_amostra].head(15).to_string(index=False))
    
    print("\n" + "="*80)
    print("SALVANDO ARQUIVO")
    print("="*80)
    
    df_final.to_csv(OUTPUT_PATH, index=False)
    print(f"\n✅ Arquivo salvo: {OUTPUT_PATH}")
    print(f"📊 Dimensões: {len(df_final)} registros × {len(df_final.columns)} colunas")
    print(f"   ⚠️  nota_aluno removida, apenas dificuldade (individual do .log)")
    
    return df_final


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    df = combinar_metricas()
    
    if df is not None:
        print("\n" + "="*80)
        print("✅ PROCESSAMENTO CONCLUÍDO COM SUCESSO!")
        print("="*80)